# DIMER Workshop: Building a Two-Stage Semantic Search and Reranking Pipeline

**Profile:** `TASK-INFERENCE` · **Mode:** `WORKSHOP` · **Notebook Specification:** 2.1 · **Standalone:** yes

This workshop composes two **live DIMER language models** into a retrieval system:

1. **Qwen3-Embedding-0.6B** retrieves a small candidate set from the full document corpus.
2. **Qwen3-Reranker-0.6B** scores only those candidates and reorders them.

The default exercise uses Banking77 customer-support messages as queries and the 77 intent phrases as the document corpus. The correct intent phrase is the single relevant document for each query.

This is not a model-vs-model leaderboard: the two models have different jobs. The central systems question is:

> **Does reranking the embedding model's top candidates improve final ordering, and what errors can reranking never repair because retrieval omitted the relevant document?**

No model is fine-tuned. The notebook reports the retriever's candidate-recall ceiling explicitly.


## Learning goals

You will:

1. distinguish **candidate retrieval** from **candidate reranking**;
2. understand why a reranker cannot recover a relevant document that the retriever did not surface;
3. compare a lexical floor, embedding-only ranking, and embedding→reranker two-stage ranking;
4. interpret Recall@k, MRR, nDCG@10, median rank, and conditional reranker accuracy;
5. measure retrieval and reranking latency separately; and
6. export per-query rankings and immutable model provenance for downstream analysis.

The optional generation stage from the earlier design is intentionally **not on the default path**. Answer generation is a third task with different evidence requirements; it is left as an extension after retrieval quality is understood.


In [ ]:
# @title 0. Workshop controls and runtime checks
USE_BYOD = False  # @param {type:"boolean"}
BYOD_ZIP_PATH = ""  # @param {type:"string"}
CANDIDATE_K = 10  # @param {type:"integer"}
MAX_QUERIES = 385  # @param {type:"integer"}
SHOW_EXAMPLES = 10  # @param {type:"integer"}

import gc
import hashlib
import importlib.metadata
import json
import math
import os
import platform
import re
import stat
import subprocess
import sys
import time
import venv
import zipfile
from pathlib import Path, PurePosixPath

from packaging.version import Version

if sys.version_info[:2] != (3,12):
    raise RuntimeError(f"This reviewed workshop supports Python 3.12; found {platform.python_version()}.")

def require_version(name,minimum,maximum):
    value=Version(importlib.metadata.version(name))
    if not (Version(minimum)<=value<Version(maximum)):
        raise RuntimeError(f"{name} {value} is outside [{minimum}, {maximum}).")
    return str(value)

CONTROL_VERSIONS={
    "numpy":require_version("numpy","1.26","3.0"),
    "pandas":require_version("pandas","2.2","4.0"),
    "matplotlib":require_version("matplotlib","3.8","4.0"),
    "packaging":require_version("packaging","24.0","27.0"),
}
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if CANDIDATE_K<2 or CANDIDATE_K>32:
    raise ValueError("CANDIDATE_K must be in 2..32 so one query shortlist fits the reranker's public batch ceiling.")
if MAX_QUERIES<10 or MAX_QUERIES>2000:
    raise ValueError("MAX_QUERIES must be in 10..2000.")

def gpu_info():
    try:
        return subprocess.check_output(
            ["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader,nounits"],
            text=True,stderr=subprocess.STDOUT
        ).strip()
    except Exception:
        return ""

GPU_INFO=gpu_info()
if not GPU_INFO:
    raise RuntimeError("Select a CUDA GPU runtime before Run all; both 0.6B models are GPU-oriented in this workshop.")

ROOT=Path.cwd()
RUNTIME_ROOT=ROOT/"workshop_runtime"
EMBEDDED_ROOT=RUNTIME_ROOT/"embedded_src"
OUTPUT_ROOT=ROOT/"outputs"
for path in (RUNTIME_ROOT,EMBEDDED_ROOT,OUTPUT_ROOT):
    path.mkdir(parents=True,exist_ok=True)

print({"python":platform.python_version(),"control_versions":CONTROL_VERSIONS,"gpu":GPU_INFO})


## 1. Exact model and implementation provenance

The live DIMER identities are:

| Stage | Model | Immutable revision | Output used here |
|---|---|---|---|
| Retrieval | `Qwen/Qwen3-Embedding-0.6B` | `97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3` | 1024-d L2-normalized embedding; cosine ranking |
| Reranking | `Qwen/Qwen3-Reranker-0.6B` | `e61197ed45024b0ed8a2d74b80b4d909f1255473` | query-document relevance score; used only to order a shortlist |

Both are Apache-2.0 and share the same reviewed PyTorch/Transformers stack.

The notebook embeds the DIMER package surfaces at authoring time from:

- `qwen3-embedding-pipeline@115cf17fb35048dcadc187ac7dd36b28d99f9aab`
- `qwen3-reranker-pipeline@f13a58e65a7ee54343e8fa262c166308699c4f11`

The packages and snapshot manifests are materialized locally. Runtime never clones a repository or downloads DIMER source.


In [ ]:
# @title 1.1 Materialize embedded package sources and manifests
EMBEDDED_SOURCES={"qwen3_embedding_pipeline/__init__.py":"from .metrics import (\n    METRIC_DEFINITIONS,\n    jaccard,\n    lexical_baseline,\n    random_floor,\n    rank_of_positive,\n    retrieval_metrics,\n)\nfrom .pipeline import (\n    ARTIFACT_FORMAT,\n    DECODER_LAYERS,\n    DEFAULT_QUERY_INSTRUCTION,\n    DEFAULT_TEMPERATURE,\n    DEFAULT_TRAINABLE_LAYERS,\n    DEFAULT_WEIGHTS_DIR,\n    EMBEDDING_DIM,\n    INPUT_SCHEMA,\n    MAX_BATCH,\n    MAX_TEXT_CHARS,\n    MAX_TEXT_TOKENS,\n    MAX_TRAIN_TOKENS,\n    MODEL_ID,\n    MODEL_KEY,\n    MODEL_LICENSE,\n    MODEL_REVISION,\n    PARAMETER_COUNT,\n    WEIGHT_FILE,\n    WEIGHT_SHA256,\n    Qwen3EmbeddingPipeline,\n    cosine_similarity,\n    evaluation_report,\n    format_query,\n    stage_missing_files,\n    validate_inputs,\n    verify_snapshot,\n)\nfrom .samples import (\n    CORPUS_FILES,\n    CORPUS_LICENSE,\n    CORPUS_NAME,\n    DEFAULT_INSTRUCTION,\n    SAMPLE_SPLIT,\n    build_sample_dataset,\n    check_split_disjoint,\n    dataset_digest,\n    documents,\n    fetch_sample_dataset,\n    intent_phrase,\n    load_byod_dataset,\n    split_dataset,\n    validate_dataset,\n    write_dataset_csv,\n)\n\n__all__ = [\n    \"ARTIFACT_FORMAT\",\n    \"CORPUS_FILES\",\n    \"CORPUS_LICENSE\",\n    \"CORPUS_NAME\",\n    \"DECODER_LAYERS\",\n    \"DEFAULT_INSTRUCTION\",\n    \"DEFAULT_QUERY_INSTRUCTION\",\n    \"DEFAULT_TEMPERATURE\",\n    \"DEFAULT_TRAINABLE_LAYERS\",\n    \"DEFAULT_WEIGHTS_DIR\",\n    \"EMBEDDING_DIM\",\n    \"INPUT_SCHEMA\",\n    \"MAX_BATCH\",\n    \"MAX_TEXT_CHARS\",\n    \"MAX_TEXT_TOKENS\",\n    \"MAX_TRAIN_TOKENS\",\n    \"METRIC_DEFINITIONS\",\n    \"MODEL_ID\",\n    \"MODEL_KEY\",\n    \"MODEL_LICENSE\",\n    \"MODEL_REVISION\",\n    \"PARAMETER_COUNT\",\n    \"SAMPLE_SPLIT\",\n    \"WEIGHT_FILE\",\n    \"WEIGHT_SHA256\",\n    \"Qwen3EmbeddingPipeline\",\n    \"build_sample_dataset\",\n    \"check_split_disjoint\",\n    \"cosine_similarity\",\n    \"dataset_digest\",\n    \"documents\",\n    \"evaluation_report\",\n    \"fetch_sample_dataset\",\n    \"format_query\",\n    \"intent_phrase\",\n    \"jaccard\",\n    \"lexical_baseline\",\n    \"load_byod_dataset\",\n    \"random_floor\",\n    \"rank_of_positive\",\n    \"retrieval_metrics\",\n    \"split_dataset\",\n    \"stage_missing_files\",\n    \"validate_dataset\",\n    \"validate_inputs\",\n    \"verify_snapshot\",\n    \"write_dataset_csv\",\n]\n","qwen3_embedding_pipeline/metrics.py":"\"\"\"Retrieval metrics over a query–positive dataset and a lexical baseline.\n\nEvery query is ranked against the dataset's document set (its sorted unique positives) by a similarity the\ncaller supplies — the embedder's cosine, or bag-of-words overlap for the baseline — and the rank of the\nquery's own positive is read: **recall@1**, **recall@5** and **recall@10** (the positive is within the first\n*k*), and **MRR** (mean of 1 / rank). Ties are resolved pessimistically (a tied document counts as ranked\nabove the positive), so a similarity that cannot separate documents scores as badly as it deserves. The\n**random floor** is what a uniformly random ranking over the document set achieves in expectation; the\n**lexical baseline** ranks documents by the Jaccard overlap of lower-cased alphanumeric tokens with the\nquery — what a system with no model at all gets from shared words.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport re\nfrom collections.abc import Mapping, Sequence\nfrom typing import Any\n\nRECALL_AT = (1, 5, 10)\nMETRIC_DEFINITIONS = {\n    \"recall@k\": (\n        \"fraction of queries whose positive document is ranked within the first k of the document set; \"\n        \"ties count against the positive\"\n    ),\n    \"mrr\": \"mean over queries of 1 / rank of the positive document\",\n    \"document_set\": \"the sorted unique positive (and explicit negative) documents of the scored dataset\",\n}\n_TOKEN_RE = re.compile(r\"[a-z0-9]+\")\n\n\ndef rank_of_positive(scores: Sequence[float], positive_index: int) -> int:\n    \"\"\"1-based rank of `positive_index` under descending `scores`; ties are ranked above the positive.\"\"\"\n    if not 0 <= positive_index < len(scores):\n        raise ValueError(\"positive_index is outside the document set\")\n    target = float(scores[positive_index])\n    return 1 + sum(1 for i, s in enumerate(scores) if i != positive_index and float(s) >= target)\n\n\ndef retrieval_metrics(ranks: Sequence[int], n_documents: int) -> dict[str, Any]:\n    \"\"\"Aggregate 1-based ranks of each query's positive into recall@k and MRR.\"\"\"\n    if not ranks:\n        raise ValueError(\"no queries to score\")\n    if n_documents < 1:\n        raise ValueError(\"n_documents must be positive\")\n    if any(not isinstance(r, int) or r < 1 or r > n_documents for r in ranks):\n        raise ValueError(\"every rank must be an int in 1..n_documents\")\n    out: dict[str, Any] = {\"n_queries\": len(ranks), \"n_documents\": n_documents}\n    for k in RECALL_AT:\n        out[f\"recall@{k}\"] = sum(1 for r in ranks if r <= k) / len(ranks)\n    out[\"mrr\"] = sum(1.0 / r for r in ranks) / len(ranks)\n    out[\"median_rank\"] = sorted(ranks)[len(ranks) // 2]\n    out[\"definitions\"] = dict(METRIC_DEFINITIONS)\n    return out\n\n\ndef random_floor(n_documents: int) -> dict[str, Any]:\n    \"\"\"Expected metrics of a uniformly random ranking over `n_documents` documents.\"\"\"\n    if n_documents < 1:\n        raise ValueError(\"n_documents must be positive\")\n    out: dict[str, Any] = {\"n_documents\": n_documents}\n    for k in RECALL_AT:\n        out[f\"recall@{k}\"] = min(k, n_documents) / n_documents\n    out[\"mrr\"] = sum(1.0 / r for r in range(1, n_documents + 1)) / n_documents\n    out[\"baseline\"] = \"uniformly random ranking of the document set (expected values)\"\n    return out\n\n\ndef _tokens(text: str) -> set[str]:\n    return set(_TOKEN_RE.findall(text.lower()))\n\n\ndef jaccard(a: str, b: str) -> float:\n    x, y = _tokens(a), _tokens(b)\n    if not x or not y:\n        return 0.0\n    return len(x & y) / len(x | y)\n\n\ndef lexical_baseline(records: Sequence[Mapping[str, Any]], documents: Sequence[str]) -> dict[str, Any]:\n    \"\"\"Rank the documents for every query by Jaccard token overlap — the no-model floor.\"\"\"\n    index = {doc: i for i, doc in enumerate(documents)}\n    ranks = []\n    for record in records:\n        if record[\"positive\"] not in index:\n            raise ValueError(f\"positive {record['positive']!r} is not in the document set\")\n        scores = [jaccard(record[\"query\"], doc) for doc in documents]\n        ranks.append(rank_of_positive(scores, index[record[\"positive\"]]))\n    result = retrieval_metrics(ranks, len(documents))\n    result[\"baseline\"] = \"Jaccard overlap of lower-cased alphanumeric tokens between query and document\"\n    return result\n","qwen3_embedding_pipeline/pipeline.py":"from __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nimport time\nfrom collections.abc import Callable, Mapping, Sequence\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\n\nMODEL_ID = \"Qwen/Qwen3-Embedding-0.6B\"\nMODEL_REVISION = \"97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3\"\nMODEL_LICENSE = \"apache-2.0\"\nMODEL_KEY = \"qwen3-embedding-0.6b\"\nDEFAULT_WEIGHTS_DIR = Path(__file__).resolve().parents[2] / \"weights\" / MODEL_KEY\nMANIFEST_NAME = \"dimer-base-manifest.json\"\n\n# Contract from the pinned upstream README (\"Transformers Usage\"): left padding, last-token pooling,\n# L2 normalisation, max_length 8192, and an \"Instruct: ...\\nQuery:\" prefix on queries only.\nEMBEDDING_DIM = 1024  # hidden_size in the pinned config.json; 1_Pooling/config.json word_embedding_dimension\nMAX_TEXT_TOKENS = 8192  # tokenizer truncation length; the model's context is 32768 but the README uses 8192\nMAX_TEXT_CHARS = 100_000  # pre-tokenisation guard so a runaway string is rejected before it is tokenised\nMAX_BATCH = 64  # texts per embed() call\nDEFAULT_QUERY_INSTRUCTION = \"Given a web search query, retrieve relevant passages that answer the query\"\nPOOLING = \"last_token\"\nKINDS = (\"query\", \"document\")\nWEIGHT_FILE = \"model.safetensors\"\nWEIGHT_SHA256 = (\n    \"0437e45c94563b09e13cb7a64478fc406947a93cb34a7e05870fc8dcd48e23fd\"  # manifest digest of WEIGHT_FILE\n)\nPARAMETER_COUNT = 595_776_512  # Qwen3Model (no LM head)\nDECODER_LAYERS = 28  # config.json num_hidden_layers\nDEFAULT_TRAINABLE_LAYERS = 2  # the last two decoder layers (31,461,888 parameters)\nMAX_TRAIN_TOKENS = (\n    64  # training-only truncation of queries and documents (inference truncates at MAX_TEXT_TOKENS)\n)\nDEFAULT_TEMPERATURE = 0.05\nMAX_EVAL_RECORDS = 2_000\nMAX_DOCUMENTS = 1_000\nMIN_SCORED_RECORDS = 50  # below this a scored dataset is labelled a small sample\nARTIFACT_FORMAT = \"org.valcorza.qwen3-embedding-0.6b.adapter.v1\"\nARTIFACT_FORMAT_VERSION = \"1.0\"\nARTIFACT_WEIGHTS_NAME = \"adapter.safetensors\"\nARTIFACT_MANIFEST_NAME = \"manifest.json\"\n\n\ndef _sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with open(path, \"rb\") as handle:\n        for chunk in iter(lambda: handle.read(1 << 20), b\"\"):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:\n    \"\"\"Check the local snapshot against its DIMER manifest; raise naming the first mismatch.\"\"\"\n    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR\n    manifest_path = root / MANIFEST_NAME\n    if not manifest_path.is_file():\n        raise FileNotFoundError(f\"snapshot manifest not found: {manifest_path}\")\n    manifest = json.loads(manifest_path.read_text(encoding=\"utf-8\"))\n    if manifest.get(\"modelId\") != MODEL_ID:\n        raise ValueError(f\"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}\")\n    if manifest.get(\"revision\") != MODEL_REVISION:\n        raise ValueError(f\"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}\")\n    for entry in manifest[\"files\"]:\n        file_path = root / entry[\"path\"]\n        if not file_path.is_file():\n            raise FileNotFoundError(f\"snapshot file missing: {file_path}\")\n        size = file_path.stat().st_size\n        if size != entry[\"bytes\"]:\n            raise ValueError(f\"{entry['path']}: size {size} != manifest {entry['bytes']}\")\n        digest = hashlib.sha256()\n        with open(file_path, \"rb\") as fh:\n            for chunk in iter(lambda: fh.read(1 << 20), b\"\"):\n                digest.update(chunk)\n        if digest.hexdigest() != entry[\"sha256\"]:\n            raise ValueError(f\"{entry['path']}: sha256 {digest.hexdigest()} != manifest {entry['sha256']}\")\n    return manifest\n\n\ndef _hub_download(relative_path: str, root: Path) -> None:\n    \"\"\"Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory.\"\"\"\n    from huggingface_hub import hf_hub_download\n\n    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))\n\n\ndef stage_missing_files(\n    path: str | Path | None = None,\n    *,\n    allow_download: bool = False,\n    downloader: Callable[[str, Path], None] | None = None,\n) -> list[str]:\n    \"\"\"Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but\n    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after.\"\"\"\n    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR\n    manifest_path = root / MANIFEST_NAME\n    if not manifest_path.is_file():\n        raise FileNotFoundError(f\"manifest not found: {manifest_path}\")\n    with open(manifest_path, encoding=\"utf-8\") as fh:\n        manifest = json.load(fh)\n    if manifest.get(\"modelId\") != MODEL_ID or manifest.get(\"revision\") != MODEL_REVISION:\n        raise ValueError(\n            f\"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, \"\n            f\"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage\"\n        )\n    missing = [entry[\"path\"] for entry in manifest[\"files\"] if not (root / entry[\"path\"]).is_file()]\n    if not missing:\n        return []\n    if not allow_download:\n        raise FileNotFoundError(\n            f\"snapshot at {root} is missing {missing}; \"\n            f\"pass allow_download=True to fetch them at {MODEL_REVISION}\"\n        )\n    fetch = downloader or _hub_download\n    for relative_path in missing:\n        fetch(relative_path, root)\n    return missing\n\n\ndef format_query(query: str, instruction: str = DEFAULT_QUERY_INSTRUCTION) -> str:\n    \"\"\"Upstream `get_detailed_instruct`: queries carry a task instruction, documents do not.\"\"\"\n    return f\"Instruct: {instruction}\\nQuery:{query}\"\n\n\ndef cosine_similarity(a: Sequence[Sequence[float]], b: Sequence[Sequence[float]]) -> list[list[float]]:\n    \"\"\"Cosine similarity matrix between two lists of vectors (no metric: there is no ground truth).\"\"\"\n    x = np.asarray(a, dtype=np.float32)\n    y = np.asarray(b, dtype=np.float32)\n    if x.ndim != 2 or y.ndim != 2 or x.shape[1] != y.shape[1]:\n        raise ValueError(\"inputs must be 2-D with the same embedding dimension\")\n    x = x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-12)\n    y = y / np.maximum(np.linalg.norm(y, axis=1, keepdims=True), 1e-12)\n    return (x @ y.T).tolist()\n\n\nINPUT_SCHEMA: dict[str, Any] = {\n    \"input\": \"sequence of non-empty str; one vector is returned per text, in input order\",\n    \"batch\": [1, MAX_BATCH],\n    \"text_chars\": [1, MAX_TEXT_CHARS],\n    \"text_tokens\": [1, MAX_TEXT_TOKENS],\n    \"kind\": list(KINDS),\n    \"embedding_dim\": EMBEDDING_DIM,\n    \"preprocessing\": (\n        \"left-padded tokenisation truncated at MAX_TEXT_TOKENS; kind='query' prepends \"\n        \"'Instruct: <instruction>\\\\nQuery:'; last-token pooling, then L2 normalisation\"\n    ),\n}\n\n\ndef _check_inputs(texts: Any, kind: str, instruction: str) -> list[str]:\n    \"\"\"Raise TypeError/ValueError naming the first violated ceiling; return the texts as a list.\"\"\"\n    if isinstance(texts, str | bytes) or not isinstance(texts, Sequence):\n        raise TypeError(\"texts must be a list of str, not a single string\")\n    if not 1 <= len(texts) <= MAX_BATCH:\n        raise ValueError(f\"texts must hold 1..{MAX_BATCH} items, got {len(texts)}\")\n    for i, text in enumerate(texts):\n        if not isinstance(text, str):\n            raise TypeError(f\"texts[{i}] must be str, got {type(text).__name__}\")\n        if not text.strip():\n            raise ValueError(f\"texts[{i}] is empty\")\n        if len(text) > MAX_TEXT_CHARS:\n            raise ValueError(f\"texts[{i}] has {len(text)} chars; ceiling is {MAX_TEXT_CHARS}\")\n    if kind not in KINDS:\n        raise ValueError(f\"kind must be one of {KINDS}\")\n    if not isinstance(instruction, str) or not instruction.strip():\n        raise ValueError(\"instruction must be a non-empty str\")\n    return list(texts)\n\n\ndef validate_inputs(\n    texts: Sequence[str],\n    kind: str = \"document\",\n    instruction: str = DEFAULT_QUERY_INSTRUCTION,\n    *,\n    names: Sequence[str] | None = None,\n) -> dict[str, Any]:\n    \"\"\"Validation stage: return the input manifest (schema, per-input observations, verdict).\n\n    Rejection is reported by raising exactly as ``embed`` would — both route through\n    ``_check_inputs`` — so a caller that wants the finding recorded catches the exception and\n    stores ``str(exc)`` under ``findings``. Token-level truncation cannot be observed here\n    because it happens inside the tokenizer; ``embed`` reports it in ``truncated``.\n    \"\"\"\n    checked = _check_inputs(texts, kind, instruction)\n    if names is not None and len(names) != len(checked):\n        raise ValueError(\"names must have one entry per text\")\n    return {\n        \"schema\": dict(INPUT_SCHEMA),\n        \"inputs\": [\n            {\"id\": names[i] if names else f\"text-{i}\", \"chars\": len(text), \"kind\": kind}\n            for i, text in enumerate(checked)\n        ],\n        \"kind\": kind,\n        \"instruction\": instruction if kind == \"query\" else None,\n        \"verdict\": \"accepted\",\n        \"findings\": [],\n        \"model_id\": MODEL_ID,\n        \"model_revision\": MODEL_REVISION,\n    }\n\n\ndef evaluation_report(\n    result: Mapping[str, Any], labels: Sequence[Any] | None = None, *, sample_kind: str = \"synthetic\"\n) -> dict[str, Any]:\n    \"\"\"Evaluation stage: a machine-readable report even though no metric exists here.\n\n    Embeddings are representations, so the repository ships no performance metric —\n    ``cosine_similarity`` is a comparison helper, not a score against ground truth. The verdict\n    is therefore always ``not-measurable`` (EVAL9), including when ``labels`` is supplied:\n    the parameter exists for interface parity with the fleet's other pipelines and is recorded\n    in ``reason`` rather than scored.\n    \"\"\"\n    embeddings = result[\"embeddings\"]\n    supplied = labels is not None\n    return {\n        \"task\": \"text embedding (dense representation, no label space)\",\n        \"score_semantics\": (\n            f\"{EMBEDDING_DIM}-d unit-norm vectors, {POOLING} pooling; cosine between two vectors of \"\n            \"this model is a similarity in [-1, 1], not a probability and not calibrated\"\n        ),\n        \"sample_kind\": sample_kind,\n        \"n_texts\": len(embeddings),\n        \"metrics\": [],\n        \"baselines\": [],\n        \"verdict\": \"not-measurable\",\n        \"reason\": (\n            \"the output is a representation, not a prediction: the pipeline exposes no performance \"\n            \"metric, only the cosine_similarity comparison helper\"\n            + (\"; labels were supplied but no metric helper exists to score them here\" if supplied else \"\")\n        ),\n        \"needs\": (\n            \"a downstream labelled task: for retrieval, a query-document set with relevance \"\n            \"judgements scored by nDCG@k or recall@k; for classification or clustering, labelled \"\n            \"texts and a fitted classifier or cluster assignment — none of which this repository ships\"\n        ),\n        \"model_id\": MODEL_ID,\n        \"model_revision\": MODEL_REVISION,\n    }\n\n\n@dataclass\nclass Qwen3EmbeddingPipeline:\n    \"\"\"Text embedder. `_runner` maps formatted texts to (pooled un-normalised vectors, token counts).\"\"\"\n\n    _runner: Callable[[list[str]], tuple[np.ndarray, list[int]]]\n    device: str\n    adapter: dict[str, Any] | None = field(default=None, repr=False)\n    _model: Any = field(default=None, repr=False)\n    _tokenizer: Any = field(default=None, repr=False)\n\n    @classmethod\n    def from_pretrained(\n        cls,\n        device: str | None = None,\n        weights_dir: str | Path | None = None,\n        allow_download: bool = False,\n    ) -> Qwen3EmbeddingPipeline:\n        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR\n        if (root / MANIFEST_NAME).is_file():\n            stage_missing_files(root, allow_download=allow_download)\n            verify_snapshot(root)\n            source, kwargs = str(root), dict(local_files_only=True)\n        elif allow_download:\n            source, kwargs = MODEL_ID, dict(revision=MODEL_REVISION)\n        else:\n            raise FileNotFoundError(f\"no verified snapshot at {root} and allow_download=False\")\n        # Refuse invalid snapshots before importing model libraries.\n        import torch\n        from transformers import AutoModel, AutoTokenizer\n\n        resolved_device = device or (\"cuda:0\" if torch.cuda.is_available() else \"cpu\")\n        dtype = torch.bfloat16 if resolved_device.startswith(\"cuda\") else torch.float32\n        tokenizer = AutoTokenizer.from_pretrained(\n            source, padding_side=\"left\", trust_remote_code=False, **kwargs\n        )\n        model = AutoModel.from_pretrained(source, dtype=dtype, trust_remote_code=False, **kwargs)\n        model = model.to(resolved_device).eval()\n\n        def runner(texts: list[str]) -> tuple[np.ndarray, list[int]]:\n            batch = tokenizer(\n                texts, padding=True, truncation=True, max_length=MAX_TEXT_TOKENS, return_tensors=\"pt\"\n            )\n            batch = batch.to(resolved_device)\n            with torch.inference_mode():\n                hidden = model(**batch).last_hidden_state\n            pooled = hidden[:, -1]  # left padding: the last position is the last real token of every row\n            counts = batch[\"attention_mask\"].sum(dim=1).tolist()\n            return pooled.float().cpu().numpy(), [int(c) for c in counts]\n\n        return cls(runner, resolved_device, _model=model, _tokenizer=tokenizer)\n\n    def _validate(self, texts: Any, kind: str, instruction: str) -> list[str]:\n        return _check_inputs(texts, kind, instruction)\n\n    def embed(\n        self,\n        texts: Sequence[str],\n        kind: str = \"document\",\n        instruction: str = DEFAULT_QUERY_INSTRUCTION,\n    ) -> dict[str, Any]:\n        \"\"\"Embed up to MAX_BATCH texts. `kind=\"query\"` prepends the instruction; documents get none.\"\"\"\n        texts = self._validate(texts, kind, instruction)\n        formatted = [format_query(t, instruction) if kind == \"query\" else t for t in texts]\n        pooled, n_tokens = self._runner(formatted)\n        pooled = np.asarray(pooled, dtype=np.float32)\n        if pooled.shape != (len(texts), EMBEDDING_DIM):\n            raise RuntimeError(f\"backend returned {pooled.shape}, expected ({len(texts)}, {EMBEDDING_DIM})\")\n        normalized = pooled / np.maximum(np.linalg.norm(pooled, axis=1, keepdims=True), 1e-12)\n        return {\n            \"embeddings\": normalized.tolist(),\n            \"dim\": EMBEDDING_DIM,\n            \"pooling\": POOLING,\n            \"normalized\": True,\n            \"kind\": kind,\n            \"instruction\": instruction if kind == \"query\" else None,\n            \"n_tokens\": list(n_tokens),\n            \"truncated\": [n >= MAX_TEXT_TOKENS for n in n_tokens],\n            \"model_id\": MODEL_ID,\n            \"model_revision\": MODEL_REVISION,\n        }\n\n    # ---- adaptation -----------------------------------------------------------------------------------\n\n    def _require_model(self) -> tuple[Any, Any]:\n        if self._model is None or self._tokenizer is None:\n            raise ValueError(\n                \"this operation needs a pipeline built with from_pretrained() or from_artifact()\"\n            )\n        return self._model, self._tokenizer\n\n    def _embed_all(self, texts: Sequence[str], kind: str, instruction: str) -> np.ndarray:\n        \"\"\"Embed any number of texts through the public contract, MAX_BATCH at a time.\"\"\"\n        rows = []\n        for start in range(0, len(texts), MAX_BATCH):\n            rows.extend(\n                self.embed(list(texts[start : start + MAX_BATCH]), kind=kind, instruction=instruction)[\n                    \"embeddings\"\n                ]\n            )\n        return np.asarray(rows, dtype=np.float32)\n\n    def evaluate(\n        self,\n        records: Sequence[Mapping[str, Any]],\n        *,\n        instruction: str = DEFAULT_QUERY_INSTRUCTION,\n        candidates: Sequence[str] | None = None,\n    ) -> dict[str, Any]:\n        \"\"\"Retrieval over the dataset's document set: every query (embedded with `instruction`) is ranked\n        against every document by cosine and the rank of its own positive is read (recall@k, MRR).\"\"\"\n        from .metrics import rank_of_positive, retrieval_metrics\n        from .samples import documents, validate_dataset\n\n        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)[\"records\"]\n        docs = list(candidates) if candidates is not None else documents(checked)\n        if not 2 <= len(docs) <= MAX_DOCUMENTS:\n            raise ValueError(f\"the document set must hold 2..{MAX_DOCUMENTS} documents; got {len(docs)}\")\n        if len(set(docs)) != len(docs):\n            duplicate = next(d for d in docs if docs.count(d) > 1)\n            raise ValueError(f\"the document set holds a duplicate candidate: {duplicate[:60]!r}\")\n        index = {doc: i for i, doc in enumerate(docs)}\n        missing = [r[\"positive\"] for r in checked if r[\"positive\"] not in index]\n        if missing:\n            raise ValueError(f\"positive {missing[0]!r} is not in the document set\")\n        started = time.perf_counter()\n        doc_vectors = self._embed_all(docs, \"document\", instruction)\n        query_vectors = self._embed_all([r[\"query\"] for r in checked], \"query\", instruction)\n        scores = query_vectors @ doc_vectors.T\n        ranks = [\n            rank_of_positive(row.tolist(), index[r[\"positive\"]])\n            for row, r in zip(scores, checked, strict=True)\n        ]\n        metrics = retrieval_metrics(ranks, len(docs))\n        metrics.update(\n            {\n                \"instruction\": instruction,\n                \"verdict\": \"measured\" if len(checked) >= MIN_SCORED_RECORDS else \"measured-small-sample\",\n                \"adapted\": self.adapter is not None,\n                \"seconds\": round(time.perf_counter() - started, 3),\n                \"model_id\": MODEL_ID,\n                \"model_revision\": MODEL_REVISION,\n            }\n        )\n        return metrics\n\n    @staticmethod\n    def lexical_baseline(\n        records: Sequence[Mapping[str, Any]], candidates: Sequence[str] | None = None\n    ) -> dict[str, Any]:\n        \"\"\"The no-model floor: documents ranked by token overlap with the query (see metrics.py).\"\"\"\n        from .metrics import lexical_baseline\n        from .samples import documents, validate_dataset\n\n        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)[\"records\"]\n        return lexical_baseline(checked, list(candidates) if candidates is not None else documents(checked))\n\n    def _trainable_names(self, trainable_layers: int) -> list[str]:\n        if not isinstance(trainable_layers, int) or not 1 <= trainable_layers <= DECODER_LAYERS:\n            raise ValueError(f\"trainable_layers must be an int in 1..{DECODER_LAYERS}\")\n        model, _ = self._require_model()\n        first = DECODER_LAYERS - trainable_layers\n        prefixes = tuple(f\"layers.{k}.\" for k in range(first, DECODER_LAYERS))\n        return [name for name, _p in model.named_parameters() if name.startswith(prefixes)]\n\n    def adapt(\n        self,\n        train: Sequence[Mapping[str, Any]],\n        val: Sequence[Mapping[str, Any]] | None = None,\n        *,\n        instruction: str = DEFAULT_QUERY_INSTRUCTION,\n        epochs: int = 2,\n        lr: float = 5e-5,\n        batch_size: int = 16,\n        trainable_layers: int = DEFAULT_TRAINABLE_LAYERS,\n        temperature: float = DEFAULT_TEMPERATURE,\n        seed: int = 0,\n        progress: Callable[[dict[str, Any]], None] | None = None,\n    ) -> dict[str, Any]:\n        \"\"\"Bounded contrastive fine-tuning on validated query–positive pairs.\n\n        Only the last `trainable_layers` decoder layers train (2 by default; the token embeddings, the\n        earlier layers and the final norm stay frozen). Each batch embeds its queries (formatted with\n        `instruction`) and the unique documents among its positives (plus any explicit negatives) in one\n        forward pass; the loss is the InfoNCE cross-entropy of every query over that batch's documents at\n        `temperature` — the other queries' positives are the negatives — with AdamW at a fixed learning rate,\n        gradient clipping at 1.0, seeded shuffling and no scheduler; texts are truncated to MAX_TRAIN_TOKENS\n        **during training only**. Epoch 0 records the frozen model's validation retrieval metrics against\n        the validation document set; the epoch with the highest validation MRR is kept.\"\"\"\n        from .samples import documents, validate_dataset\n\n        if not isinstance(epochs, int) or not 1 <= epochs <= 20:\n            raise ValueError(\"epochs must be an int in 1..20\")\n        if not (0.0 < lr <= 1e-3):\n            raise ValueError(\"lr must be in (0, 1e-3]\")\n        if not isinstance(batch_size, int) or not 2 <= batch_size <= 64:\n            raise ValueError(\"batch_size must be an int in 2..64\")\n        if not (0.0 < temperature <= 1.0):\n            raise ValueError(\"temperature must be in (0, 1]\")\n        _check_inputs([\"x\"], \"query\", instruction)\n        names = self._trainable_names(trainable_layers)\n        train_checked = validate_dataset(train)[\"records\"]\n        val_checked = (\n            validate_dataset(val, min_records=1, max_records=MAX_EVAL_RECORDS)[\"records\"] if val else []\n        )\n        val_docs = documents(val_checked) if val_checked else []\n        import torch\n\n        torch.manual_seed(seed)\n        model, tokenizer = self._require_model()\n        started = time.perf_counter()\n        wanted = set(names)\n        for name, param in model.named_parameters():\n            param.requires_grad_(name in wanted)\n        params = [p for p in model.parameters() if p.requires_grad]\n        n_trainable = sum(p.numel() for p in params)\n        optimiser = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)\n        device = torch.device(self.device)\n\n        def score_val() -> dict[str, Any] | None:\n            if not val_checked:\n                return None\n            model.eval()\n            keep = (\"recall@1\", \"recall@5\", \"recall@10\", \"mrr\", \"n_documents\")\n            return {\n                k: v\n                for k, v in self.evaluate(val_checked, instruction=instruction, candidates=val_docs).items()\n                if k in keep\n            }\n\n        def encode(texts: list[str]) -> torch.Tensor:\n            batch = tokenizer(\n                texts, padding=True, truncation=True, max_length=MAX_TRAIN_TOKENS, return_tensors=\"pt\"\n            )\n            hidden = model(**batch.to(device)).last_hidden_state[:, -1]\n            return torch.nn.functional.normalize(hidden.float(), dim=-1)\n\n        history: list[dict[str, Any]] = []\n        entry: dict[str, Any] = {\"epoch\": 0, \"train_loss\": None, \"val\": score_val(), \"note\": \"frozen model\"}\n        history.append(entry)\n        if progress:\n            progress(entry)\n        best_mrr = entry[\"val\"][\"mrr\"] if entry[\"val\"] else -math.inf\n        best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}\n        initial_state = {k: v.clone() for k, v in best_state.items()}\n        best_epoch = 0\n        generator = torch.Generator().manual_seed(seed)\n        try:\n            for epoch in range(1, epochs + 1):\n                model.train()\n                order = torch.randperm(len(train_checked), generator=generator).tolist()\n                losses = []\n                for start in range(0, len(order), batch_size):\n                    batch = [train_checked[i] for i in order[start : start + batch_size]]\n                    if len(batch) < 2:\n                        continue\n                    docs = documents(batch)\n                    targets = torch.tensor(\n                        [docs.index(r[\"positive\"]) for r in batch], dtype=torch.long, device=device\n                    )\n                    queries = encode([format_query(r[\"query\"], instruction) for r in batch])\n                    candidates = encode(docs)\n                    logits = queries @ candidates.T / temperature\n                    loss = torch.nn.functional.cross_entropy(logits, targets)\n                    optimiser.zero_grad(set_to_none=True)\n                    loss.backward()\n                    torch.nn.utils.clip_grad_norm_(params, 1.0)\n                    optimiser.step()\n                    losses.append(float(loss.detach()))\n                model.eval()\n                entry = {\"epoch\": epoch, \"train_loss\": sum(losses) / max(len(losses), 1), \"val\": score_val()}\n                history.append(entry)\n                if progress:\n                    progress(entry)\n                current = entry[\"val\"][\"mrr\"] if entry[\"val\"] else math.inf\n                if current > best_mrr or not entry[\"val\"]:\n                    best_mrr = current\n                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}\n                    best_epoch = epoch\n        except BaseException:\n            # Transactional: a failure in training, validation or the progress callback leaves the base\n            # exactly as it was, with every parameter frozen again.\n            restore = dict(model.state_dict())\n            restore.update(initial_state)\n            model.load_state_dict(restore, strict=True)\n            model.eval()\n            for param in model.parameters():\n                param.requires_grad_(False)\n            self.adapter = None\n            raise\n        merged = dict(model.state_dict())\n        merged.update(best_state)\n        model.load_state_dict(merged, strict=True)\n        model.eval()\n        for param in model.parameters():\n            param.requires_grad_(False)\n        self.adapter = {\n            \"objective\": \"InfoNCE over in-batch documents (contrastive)\",\n            \"instruction\": instruction,\n            \"trainable_layers\": trainable_layers,\n            \"trainable_names\": names,\n            \"n_trainable\": n_trainable,\n            \"n_total\": sum(p.numel() for p in model.parameters()),\n            \"epochs\": epochs,\n            \"best_epoch\": best_epoch,\n            \"selection\": \"highest validation MRR\" if val_checked else \"final epoch (no validation split)\",\n            \"lr\": lr,\n            \"batch_size\": batch_size,\n            \"temperature\": temperature,\n            \"max_train_tokens\": MAX_TRAIN_TOKENS,\n            \"n_train\": len(train_checked),\n            \"n_train_documents\": len(documents(train_checked)),\n            \"n_val\": len(val_checked),\n            \"seed\": seed,\n            \"history\": history,\n            \"seconds\": round(time.perf_counter() - started, 2),\n        }\n        return dict(self.adapter)\n\n    # ---- artifacts ------------------------------------------------------------------------------------\n\n    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:\n        \"\"\"Write the adapted decoder-layer tensors as safetensors with a manifest naming the base.\"\"\"\n        if self.adapter is None:\n            raise ValueError(\"nothing to save: call adapt() first\")\n        model, _ = self._require_model()\n        from safetensors.torch import save_file\n\n        out = Path(output_dir)\n        out.mkdir(parents=True, exist_ok=True)\n        names = set(self.adapter[\"trainable_names\"])\n        tensors = {k: v.detach().cpu().contiguous() for k, v in model.state_dict().items() if k in names}\n        weights_path = out / ARTIFACT_WEIGHTS_NAME\n        save_file(tensors, str(weights_path), metadata={\"format\": \"pt\"})\n        manifest = {\n            \"format\": ARTIFACT_FORMAT,\n            \"format_version\": ARTIFACT_FORMAT_VERSION,\n            \"base_model\": {\n                \"id\": MODEL_ID,\n                \"revision\": MODEL_REVISION,\n                \"key\": MODEL_KEY,\n                \"weight_file\": WEIGHT_FILE,\n                \"weight_sha256\": WEIGHT_SHA256,\n            },\n            \"adapter\": {k: v for k, v in self.adapter.items() if k not in (\"history\", \"trainable_names\")},\n            \"history\": self.adapter[\"history\"],\n            \"tensors\": sorted(tensors),\n            \"files\": [\n                {\n                    \"path\": ARTIFACT_WEIGHTS_NAME,\n                    \"bytes\": weights_path.stat().st_size,\n                    \"sha256\": _sha256(weights_path),\n                }\n            ],\n            \"metadata\": dict(metadata or {}),\n        }\n        (out / ARTIFACT_MANIFEST_NAME).write_text(\n            json.dumps(manifest, indent=2, ensure_ascii=False), encoding=\"utf-8\"\n        )\n        return out\n\n    def _check_artifact_manifest(self, root: Path, manifest: Mapping[str, Any]) -> Path:\n        \"\"\"Refuse an artifact whose manifest is not exactly the one this pipeline writes: the supported\n        format and version, the pinned base (id, revision, weight file, digest), exactly one file entry\n        named `adapter.safetensors` that resolves inside the artifact directory, and a recorded\n        `trainable_layers` in range. Nothing is deserialised here. The digest check that follows\n        detects corruption or drift of the weights relative to the adjacent manifest; it is not\n        authenticity against an actor who can replace both files.\"\"\"\n        if manifest.get(\"format\") != ARTIFACT_FORMAT:\n            raise ValueError(f\"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}\")\n        if manifest.get(\"format_version\") != ARTIFACT_FORMAT_VERSION:\n            raise ValueError(\n                f\"artifact format_version {manifest.get('format_version')!r} is not the supported \"\n                f\"{ARTIFACT_FORMAT_VERSION!r}\"\n            )\n        base = manifest.get(\"base_model\", {})\n        if (base.get(\"id\"), base.get(\"revision\"), base.get(\"weight_sha256\")) != (\n            MODEL_ID,\n            MODEL_REVISION,\n            WEIGHT_SHA256,\n        ):\n            raise ValueError(\"artifact was adapted from a different base model, revision or weight file\")\n        if base.get(\"weight_file\", WEIGHT_FILE) != WEIGHT_FILE:\n            raise ValueError(\"artifact was adapted from a different base weight file\")\n        files = manifest.get(\"files\")\n        if not isinstance(files, list) or len(files) != 1:\n            raise ValueError(\"artifact manifest must list exactly one file\")\n        entry = files[0]\n        if not isinstance(entry, Mapping) or entry.get(\"path\") != ARTIFACT_WEIGHTS_NAME:\n            raise ValueError(f\"artifact manifest must name exactly {ARTIFACT_WEIGHTS_NAME!r}\")\n        weights_path = (root / entry[\"path\"]).resolve()\n        if weights_path.parent != root.resolve():\n            raise ValueError(\"artifact weight path must resolve inside the artifact directory\")\n        adapter = manifest.get(\"adapter\")\n        layers = adapter.get(\"trainable_layers\") if isinstance(adapter, Mapping) else None\n        if isinstance(layers, bool) or not isinstance(layers, int):\n            raise ValueError(\"artifact manifest does not record an integer trainable_layers\")\n        if not isinstance(manifest.get(\"tensors\"), list):\n            raise ValueError(\"artifact manifest must list its tensors\")\n        return weights_path\n\n    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:\n        \"\"\"Verify an adapter's manifest, digest and exact tensor set **before** deserialising, then overwrite\n        exactly the tensors it carries.\"\"\"\n        root = Path(artifact_dir)\n        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding=\"utf-8\"))\n        weights_path = self._check_artifact_manifest(root, manifest)\n        entry = manifest[\"files\"][0]\n        if not weights_path.is_file():\n            raise FileNotFoundError(f\"artifact weights missing: {weights_path}\")\n        if _sha256(weights_path) != entry[\"sha256\"] or weights_path.stat().st_size != entry[\"bytes\"]:\n            raise ValueError(f\"{entry['path']}: digest or size mismatch; refusing to load\")\n        # The exact tensor set the recorded configuration implies — no subset, no extra, no other layer.\n        expected = sorted(self._trainable_names(manifest[\"adapter\"][\"trainable_layers\"]))\n        if sorted(manifest[\"tensors\"]) != expected:\n            raise ValueError(\"artifact tensor list does not match its recorded configuration\")\n        model, _ = self._require_model()\n        from safetensors.torch import load_file\n\n        tensors = load_file(str(weights_path))\n        if sorted(tensors) != expected:\n            raise ValueError(\"artifact tensor names differ from its manifest\")\n        state = model.state_dict()\n        for key, value in tensors.items():\n            if key not in state or not key.startswith(\"layers.\"):\n                raise ValueError(\n                    f\"artifact tensor {key} is not an adaptable decoder-layer tensor of the base\"\n                )\n            if tuple(value.shape) != tuple(state[key].shape):\n                raise ValueError(\n                    f\"artifact tensor {key}: shape {tuple(value.shape)} != {tuple(state[key].shape)}\"\n                )\n        merged = dict(state)\n        merged.update({k: v.to(state[k].dtype) for k, v in tensors.items()})\n        model.load_state_dict(merged, strict=True)\n        model.eval()\n        self.adapter = {\n            **manifest[\"adapter\"],\n            \"trainable_names\": manifest[\"tensors\"],\n            \"history\": manifest.get(\"history\", []),\n        }\n        return manifest\n\n    @classmethod\n    def from_artifact(\n        cls,\n        artifact_dir: str | Path,\n        *,\n        device: str | None = None,\n        weights_dir: str | Path | None = None,\n        allow_download: bool = False,\n    ) -> Qwen3EmbeddingPipeline:\n        pipeline = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)\n        pipeline.load_artifact(artifact_dir)\n        return pipeline\n","qwen3_embedding_pipeline/samples.py":"\"\"\"Query–positive pair dataset contract for contrastive adaptation of the embedder: the pinned Banking77\nsample, validation, seeded splitting, BYOD loaders and CSV export.\n\nThe default dataset is **real** and a retrieval task the embedder was not tuned for: Banking77 (Casanueva et\nal., 2020; CC BY 4.0), 13,083 customer-support messages labelled with 77 fine-grained banking intents. Two CSV\nfiles (`train.csv`, `test.csv`) are fetched from the PolyAI `task-specific-datasets` repository at a pinned\ncommit and refused on any byte-size or SHA-256 mismatch. Every intent name becomes a short **document**\n(`card_arrival` → `card arrival`); each message is a **query** whose positive document is its intent phrase,\nso retrieval over the 77 documents is intent detection by nearest neighbour. Training and validation queries\nare drawn from `train.csv`, test queries from `test.csv` — the release's own partition — balanced over the\n77 intents.\n\nA record is ``{id, query, positive}`` (an optional ``negative`` is accepted and carried); the document set\nof a dataset is its sorted unique positives.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport io\nimport json\nimport random\nimport re\nimport urllib.request\nfrom collections.abc import Mapping, Sequence\nfrom pathlib import Path\nfrom typing import Any\n\nfrom .pipeline import MAX_TEXT_CHARS, MODEL_ID\n\nCORPUS_NAME = \"Banking77 (messages → intent phrases)\"\nCORPUS_RELEASE = \"PolyAI-LDN/task-specific-datasets @ 57ec275d8078af65b7731c2a98be812d844a6d6b\"\nCORPUS_BASE_URL = (\n    \"https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/\"\n    \"57ec275d8078af65b7731c2a98be812d844a6d6b/banking_data/\"\n)\nCORPUS_FILES = {\n    \"train\": (\"train.csv\", 839_073, \"b06e26ac675513959a63135f11b94ea7786ed02da65db93a5650d8838cbc664b\"),\n    \"test\": (\"test.csv\", 239_961, \"d12d6e3bc4c3103966ae786dc435913c0c563dfa328f5a3646d0e62cfeeb474d\"),\n}\nCORPUS_LICENSE = \"CC BY 4.0 (Casanueva et al. 2020; PolyAI-LDN/task-specific-datasets)\"\nCORPUS_ROWS = {\"train\": 10_003, \"test\": 3_080}\nCORPUS_INTENTS = 77\nDEFAULT_CACHE_DIR = Path(\"weights\") / \"banking77\"\nDEFAULT_INSTRUCTION = \"Given a customer support message, retrieve the banking intent it expresses\"\nSAMPLE_SEED = 42\nSAMPLE_SPLIT = {\"train\": 616, \"validation\": 154, \"test\": 385}  # 8 / 2 / 5 per intent, balanced over 77\nMIN_RECORDS = 8\nMAX_RECORDS = 20_000\nMAX_DOCUMENT_CHARS = 1_000\nMIN_DOCUMENTS = 2\n_ID_RE = re.compile(r\"^[A-Za-z0-9_.:-]{1,64}$\")\n\n\ndef _sha256_bytes(data: bytes) -> str:\n    return hashlib.sha256(data).hexdigest()\n\n\ndef intent_phrase(intent: str) -> str:\n    \"\"\"The document text of an intent: its snake_case name as words (`card_arrival` → `card arrival`).\"\"\"\n    return \" \".join(intent.strip().split(\"_\"))\n\n\ndef fetch_corpus(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> dict[str, bytes]:\n    \"\"\"Return the two pinned Banking77 CSVs (bytes) from the cache or the project repository, verified.\"\"\"\n    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR\n    cache.mkdir(parents=True, exist_ok=True)\n    out = {}\n    for split, (name, size, digest) in CORPUS_FILES.items():\n        local = cache / name\n        data = local.read_bytes() if local.is_file() else b\"\"\n        if len(data) != size or _sha256_bytes(data) != digest:\n            url = CORPUS_BASE_URL + name\n            if fetcher is not None:\n                data = fetcher(url)\n            else:\n                with urllib.request.urlopen(url, timeout=120) as response:  # noqa: S310 (pinned https URL)\n                    data = response.read()\n            if len(data) != size or _sha256_bytes(data) != digest:\n                raise ValueError(\n                    f\"{name}: fetched {len(data)} bytes with sha256 {_sha256_bytes(data)[:16]}…, \"\n                    f\"pinned {size} / {digest[:16]}…\"\n                )\n            local.write_bytes(data)\n        out[split] = data\n    return out\n\n\ndef read_corpus(files: Mapping[str, bytes]) -> dict[str, list[dict[str, Any]]]:\n    \"\"\"Parse the CSV members (columns `text`, `category`) into flat records keeping the raw intent name.\"\"\"\n    out = {}\n    for split in CORPUS_FILES:\n        if split not in files:\n            raise ValueError(f\"corpus is missing the {split} file\")\n        rows = list(csv.DictReader(io.StringIO(files[split].decode(\"utf-8\"))))\n        if not rows or {\"text\", \"category\"} - set(rows[0]):\n            raise ValueError(f\"{split}: expected columns text and category\")\n        if len(rows) != CORPUS_ROWS[split]:\n            raise ValueError(f\"{split}: {len(rows)} rows, expected {CORPUS_ROWS[split]}\")\n        out[split] = [\n            {\"id\": f\"{split}-{i:05d}\", \"text\": r[\"text\"].strip(), \"intent\": r[\"category\"].strip()}\n            for i, r in enumerate(rows)\n        ]\n        intents = {r[\"intent\"] for r in out[split]}\n        if len(intents) != CORPUS_INTENTS:\n            raise ValueError(f\"{split}: {len(intents)} intents, expected {CORPUS_INTENTS}\")\n    return out\n\n\ndef filter_records(records: Sequence[Mapping[str, Any]]) -> list[dict[str, Any]]:\n    \"\"\"Turn corpus rows into query–positive pairs; drop empty, over-long and repeated queries.\"\"\"\n    seen: set[str] = set()\n    kept = []\n    for record in records:\n        query = str(record[\"text\"]).strip()\n        key = query.lower()\n        if not query or key in seen or len(query) > MAX_TEXT_CHARS:\n            continue\n        seen.add(key)\n        intent = str(record[\"intent\"])\n        kept.append({\"id\": record[\"id\"], \"query\": query, \"positive\": intent_phrase(intent), \"intent\": intent})\n    return kept\n\n\ndef build_sample_dataset(\n    corpus: Mapping[str, Sequence[Mapping[str, Any]]],\n    *,\n    seed: int = SAMPLE_SEED,\n    sizes: Mapping[str, int] | None = None,\n) -> dict[str, list[dict[str, Any]]]:\n    \"\"\"Balanced seeded draws over all 77 intents: training and validation from `train` (disjoint queries),\n    test from `test`.\"\"\"\n    sizes = dict(sizes or SAMPLE_SPLIT)\n    for name, size in sizes.items():\n        if size % CORPUS_INTENTS:\n            raise ValueError(f\"{name} size {size} is not a multiple of the {CORPUS_INTENTS} intents\")\n    rng = random.Random(seed)\n    pools = {\"train\": filter_records(corpus[\"train\"]), \"test\": filter_records(corpus[\"test\"])}\n    intents = sorted({r[\"intent\"] for r in pools[\"train\"]})\n    by_intent = {\n        split: {intent: [r for r in pool if r[\"intent\"] == intent] for intent in intents}\n        for split, pool in pools.items()\n    }\n    for split in by_intent.values():\n        for records in split.values():\n            rng.shuffle(records)\n    cursor = dict.fromkeys(intents, 0)\n    out: dict[str, list[dict[str, Any]]] = {}\n    for name, size in sizes.items():\n        source = \"test\" if name == \"test\" else \"train\"\n        per_intent = size // CORPUS_INTENTS\n        picked = []\n        for intent in intents:\n            pool = by_intent[source][intent]\n            start = cursor[intent] if source == \"train\" else 0\n            chunk = pool[start : start + per_intent]\n            if len(chunk) < per_intent:\n                raise ValueError(\n                    f\"{name}: only {len(chunk)} records available for {intent!r}, need {per_intent}\"\n                )\n            picked.extend(chunk)\n            if source == \"train\":\n                cursor[intent] = start + per_intent\n        rng.shuffle(picked)\n        out[name] = [\n            {\"id\": f\"{name}-{i:04d}\", \"query\": r[\"query\"], \"positive\": r[\"positive\"], \"intent\": r[\"intent\"]}\n            for i, r in enumerate(picked)\n        ]\n    return out\n\n\ndef fetch_sample_dataset(\n    *,\n    cache_dir: str | Path | None = None,\n    fetcher: Any = None,\n    seed: int = SAMPLE_SEED,\n    sizes: Mapping[str, int] | None = None,\n) -> dict[str, list[dict[str, Any]]]:\n    \"\"\"The tutorial splits from the pinned corpus.\"\"\"\n    return build_sample_dataset(\n        read_corpus(fetch_corpus(cache_dir=cache_dir, fetcher=fetcher)), seed=seed, sizes=sizes\n    )\n\n\ndef _check_text(value: Any, label: str, ceiling: int) -> str:\n    if not isinstance(value, str):\n        raise ValueError(f\"{label} must be a string\")\n    if not value.strip():\n        raise ValueError(f\"{label} is empty\")\n    if len(value) > ceiling:\n        raise ValueError(f\"{label} has {len(value)} chars; ceiling is {ceiling}\")\n    return value.strip()\n\n\ndef _check_record(record: Any, index: int) -> dict[str, Any]:\n    label = f\"records[{index}]\"\n    if not isinstance(record, Mapping):\n        raise ValueError(f\"{label} must be a mapping with id/query/positive\")\n    for key in (\"id\", \"query\", \"positive\"):\n        if key not in record:\n            raise ValueError(f\"{label} is missing {key!r}\")\n    rid = record[\"id\"]\n    if not isinstance(rid, str) or not _ID_RE.match(rid):\n        raise ValueError(f\"{label}: id must match {_ID_RE.pattern}\")\n    item = {\n        \"id\": rid,\n        \"query\": _check_text(record[\"query\"], f\"{label}: query\", MAX_TEXT_CHARS),\n        \"positive\": _check_text(record[\"positive\"], f\"{label}: positive\", MAX_DOCUMENT_CHARS),\n    }\n    if record.get(\"negative\") not in (None, \"\"):\n        item[\"negative\"] = _check_text(record[\"negative\"], f\"{label}: negative\", MAX_DOCUMENT_CHARS)\n        if item[\"negative\"] == item[\"positive\"]:\n            raise ValueError(f\"{label}: negative equals positive\")\n    if \"intent\" in record:\n        item[\"intent\"] = str(record[\"intent\"])\n    return item\n\n\ndef validate_dataset(\n    records: Sequence[Mapping[str, Any]], *, min_records: int = MIN_RECORDS, max_records: int = MAX_RECORDS\n) -> dict[str, Any]:\n    \"\"\"Structural validation of a query–positive dataset; raises ValueError before any model import.\"\"\"\n    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, (str, bytes)):\n        raise ValueError(\"records must be a list of {id, query, positive} mappings\")\n    if not min_records <= len(records) <= max_records:\n        raise ValueError(f\"{len(records)} records; {min_records}..{max_records} are required\")\n    checked = []\n    ids: set[str] = set()\n    queries: set[str] = set()\n    for index, record in enumerate(records):\n        item = _check_record(record, index)\n        if item[\"id\"] in ids:\n            raise ValueError(f\"duplicate id {item['id']!r}\")\n        ids.add(item[\"id\"])\n        queries.add(item[\"query\"].lower())\n        checked.append(item)\n    docs = documents(checked)\n    if len(docs) < MIN_DOCUMENTS:\n        raise ValueError(f\"a dataset needs at least {MIN_DOCUMENTS} distinct positives; found {len(docs)}\")\n    return {\n        \"records\": checked,\n        \"n_records\": len(checked),\n        \"unique_queries\": len(queries),\n        \"n_documents\": len(docs),\n        \"query_chars\": {\n            \"min\": min(len(r[\"query\"]) for r in checked),\n            \"max\": max(len(r[\"query\"]) for r in checked),\n        },\n        \"with_negative\": sum(\"negative\" in r for r in checked),\n        \"digest\": dataset_digest(checked),\n        \"model_id\": MODEL_ID,\n    }\n\n\ndef documents(records: Sequence[Mapping[str, Any]]) -> list[str]:\n    \"\"\"The sorted unique positive (and explicit negative) documents of a dataset — the candidates.\"\"\"\n    return sorted(\n        {str(r[\"positive\"]).strip() for r in records}\n        | {str(r[\"negative\"]).strip() for r in records if r.get(\"negative\")}\n    )\n\n\ndef dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:\n    payload = [[r[\"id\"], r[\"query\"], r[\"positive\"], r.get(\"negative\", \"\")] for r in records]\n    return _sha256_bytes(json.dumps(payload, ensure_ascii=False, separators=(\",\", \":\")).encode(\"utf-8\"))\n\n\ndef check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:\n    \"\"\"Assert no lower-cased query appears in two splits (leakage check).\"\"\"\n    seen: dict[str, str] = {}\n    for name, records in splits.items():\n        for record in records:\n            key = str(record[\"query\"]).lower()\n            if key in seen and seen[key] != name:\n                raise ValueError(f\"query {record['query'][:60]!r} appears in both {seen[key]} and {name}\")\n            seen[key] = name\n    return {name: len(records) for name, records in splits.items()}\n\n\ndef split_dataset(\n    records: Sequence[Mapping[str, Any]],\n    *,\n    val_fraction: float = 0.15,\n    test_fraction: float = 0.2,\n    seed: int = 0,\n) -> dict[str, list[dict[str, Any]]]:\n    \"\"\"Seeded shuffle of a BYOD dataset into train/validation/test after de-duplicating queries.\"\"\"\n    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):\n        raise ValueError(\"fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1\")\n    checked = validate_dataset(records)[\"records\"]\n    seen: set[str] = set()\n    unique = []\n    for record in checked:\n        key = record[\"query\"].lower()\n        if key not in seen:\n            seen.add(key)\n            unique.append(record)\n    random.Random(seed).shuffle(unique)\n    n_test = max(1, round(len(unique) * test_fraction))\n    n_val = round(len(unique) * val_fraction)\n    splits = {\n        \"test\": unique[:n_test],\n        \"validation\": unique[n_test : n_test + n_val],\n        \"train\": unique[n_test + n_val :],\n    }\n    if len(splits[\"train\"]) < MIN_RECORDS:\n        raise ValueError(\n            f\"split leaves {len(splits['train'])} training records; at least {MIN_RECORDS} are required\"\n        )\n    return splits\n\n\ndef load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:\n    \"\"\"Read `{id, query, positive[, negative]}` records from CSV (columns id, query, positive and an optional\n    negative), a JSON array or JSONL.\"\"\"\n    file_path = Path(path)\n    if not file_path.is_file():\n        raise FileNotFoundError(f\"dataset not found: {file_path}\")\n    suffix = file_path.suffix.lower()\n    text = file_path.read_text(encoding=\"utf-8\")\n    if suffix == \".csv\":\n        rows = list(csv.DictReader(io.StringIO(text)))\n        missing = {\"id\", \"query\", \"positive\"} - set(rows[0].keys() if rows else set())\n        if missing:\n            raise ValueError(f\"CSV is missing columns {sorted(missing)}\")\n        out = []\n        for r in rows:\n            item = {\"id\": r[\"id\"], \"query\": r[\"query\"], \"positive\": r[\"positive\"]}\n            if r.get(\"negative\"):\n                item[\"negative\"] = r[\"negative\"]\n            out.append(item)\n        return out\n    if suffix == \".jsonl\":\n        return [json.loads(line) for line in text.splitlines() if line.strip()]\n    if suffix == \".json\":\n        data = json.loads(text)\n        if not isinstance(data, list):\n            raise ValueError(\"JSON dataset must be an array of records\")\n        return data\n    raise ValueError(\"BYOD datasets must be .csv, .json or .jsonl\")\n\n\ndef write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:\n    out = Path(path)\n    out.parent.mkdir(parents=True, exist_ok=True)\n    with open(out, \"w\", encoding=\"utf-8\", newline=\"\") as handle:\n        writer = csv.DictWriter(handle, fieldnames=[\"id\", \"query\", \"positive\", \"negative\"])\n        writer.writeheader()\n        for record in records:\n            writer.writerow(\n                {\n                    \"id\": record[\"id\"],\n                    \"query\": record[\"query\"],\n                    \"positive\": record[\"positive\"],\n                    \"negative\": record.get(\"negative\", \"\"),\n                }\n            )\n    return out\n","qwen3_reranker_pipeline/__init__.py":"from .metrics import (\n    METRIC_DEFINITIONS,\n    candidate_list,\n    jaccard,\n    lexical_baseline,\n    random_floor,\n    rank_of_positive,\n    ranking_metrics,\n    record_seed,\n)\nfrom .pipeline import (\n    ARTIFACT_FORMAT,\n    DECODER_LAYERS,\n    DEFAULT_INSTRUCTION,\n    DEFAULT_TRAINABLE_LAYERS,\n    DEFAULT_WEIGHTS_DIR,\n    INPUT_SCHEMA,\n    MAX_PAIRS,\n    MAX_TEXT_CHARS,\n    MAX_TEXT_TOKENS,\n    MAX_TRAIN_CANDIDATES,\n    MAX_TRAIN_TOKENS,\n    MODEL_ID,\n    MODEL_KEY,\n    MODEL_LICENSE,\n    MODEL_REVISION,\n    PARAMETER_COUNT,\n    WEIGHT_FILE,\n    WEIGHT_SHA256,\n    Qwen3RerankerPipeline,\n    evaluation_report,\n    stage_missing_files,\n    validate_inputs,\n    verify_snapshot,\n)\nfrom .samples import (\n    CORPUS_FILES,\n    CORPUS_LICENSE,\n    CORPUS_NAME,\n    NEGATIVE_SEPARATOR,\n    SAMPLE_NEGATIVES,\n    SAMPLE_SPLIT,\n    TASK_INSTRUCTION,\n    build_sample_dataset,\n    check_split_disjoint,\n    dataset_digest,\n    documents,\n    fetch_sample_dataset,\n    intent_phrase,\n    load_byod_dataset,\n    sample_negatives,\n    split_dataset,\n    validate_dataset,\n    write_dataset_csv,\n)\n\n__all__ = [\n    \"ARTIFACT_FORMAT\",\n    \"CORPUS_FILES\",\n    \"CORPUS_LICENSE\",\n    \"CORPUS_NAME\",\n    \"DECODER_LAYERS\",\n    \"DEFAULT_INSTRUCTION\",\n    \"DEFAULT_TRAINABLE_LAYERS\",\n    \"DEFAULT_WEIGHTS_DIR\",\n    \"INPUT_SCHEMA\",\n    \"MAX_PAIRS\",\n    \"MAX_TEXT_CHARS\",\n    \"MAX_TEXT_TOKENS\",\n    \"MAX_TRAIN_CANDIDATES\",\n    \"MAX_TRAIN_TOKENS\",\n    \"METRIC_DEFINITIONS\",\n    \"MODEL_ID\",\n    \"MODEL_KEY\",\n    \"MODEL_LICENSE\",\n    \"MODEL_REVISION\",\n    \"NEGATIVE_SEPARATOR\",\n    \"PARAMETER_COUNT\",\n    \"SAMPLE_NEGATIVES\",\n    \"SAMPLE_SPLIT\",\n    \"TASK_INSTRUCTION\",\n    \"WEIGHT_FILE\",\n    \"WEIGHT_SHA256\",\n    \"Qwen3RerankerPipeline\",\n    \"build_sample_dataset\",\n    \"candidate_list\",\n    \"check_split_disjoint\",\n    \"dataset_digest\",\n    \"documents\",\n    \"evaluation_report\",\n    \"fetch_sample_dataset\",\n    \"intent_phrase\",\n    \"jaccard\",\n    \"lexical_baseline\",\n    \"load_byod_dataset\",\n    \"random_floor\",\n    \"rank_of_positive\",\n    \"ranking_metrics\",\n    \"record_seed\",\n    \"sample_negatives\",\n    \"split_dataset\",\n    \"stage_missing_files\",\n    \"validate_dataset\",\n    \"validate_inputs\",\n    \"verify_snapshot\",\n    \"write_dataset_csv\",\n]\n","qwen3_reranker_pipeline/metrics.py":"\"\"\"Ranking metrics over a query–candidate-list dataset, a random floor and a lexical baseline.\n\nEvery query comes with a candidate list — its positive document and its negatives — and a scorer ranks the\nlist; the rank of the positive gives **recall@1**, **recall@3**, **recall@5** (the positive is within the\nfirst *k*) and **MRR** (mean of 1 / rank). Ties are resolved pessimistically (a tied candidate counts as\nranked above the positive). The **random floor** is what a uniformly random ordering of each query's list\nachieves in expectation, averaged over the queries; the **lexical baseline** orders each list by the\nJaccard overlap of lower-cased alphanumeric tokens with the query — what a system with no model gets\nfrom shared words.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport hashlib\nimport re\nfrom collections.abc import Mapping, Sequence\nfrom typing import Any\n\nRECALL_AT = (1, 3, 5)\nMETRIC_DEFINITIONS = {\n    \"recall@k\": (\n        \"fraction of queries whose positive document is ranked within the first k of the query's candidate \"\n        \"list; ties count against the positive\"\n    ),\n    \"mrr\": \"mean over queries of 1 / rank of the positive document within its candidate list\",\n    \"candidates\": \"each query's list is its positive followed by its negatives; lists may differ in length\",\n}\n_TOKEN_RE = re.compile(r\"[a-z0-9]+\")\n\n\ndef record_seed(record_id: str, seed: int) -> int:\n    \"\"\"A stable per-record seed so seeded choices depend only on the record and the base seed.\"\"\"\n    digest = hashlib.sha256(f\"{seed}:{record_id}\".encode()).digest()\n    return int.from_bytes(digest[:8], \"big\")\n\n\ndef rank_of_positive(scores: Sequence[float], positive_index: int = 0) -> int:\n    \"\"\"1-based rank of `positive_index` under descending `scores`; ties are ranked above the positive.\"\"\"\n    if not 0 <= positive_index < len(scores):\n        raise ValueError(\"positive_index is outside the candidate list\")\n    target = float(scores[positive_index])\n    return 1 + sum(1 for i, s in enumerate(scores) if i != positive_index and float(s) >= target)\n\n\ndef ranking_metrics(ranks: Sequence[int], list_sizes: Sequence[int]) -> dict[str, Any]:\n    \"\"\"Aggregate 1-based ranks of each query's positive (with its list size) into recall@k and MRR.\"\"\"\n    if not ranks:\n        raise ValueError(\"no queries to score\")\n    if len(ranks) != len(list_sizes):\n        raise ValueError(\"ranks and list_sizes must align\")\n    for rank, size in zip(ranks, list_sizes, strict=True):\n        if not isinstance(size, int) or size < 2:\n            raise ValueError(\"every candidate list needs at least two entries\")\n        if not isinstance(rank, int) or not 1 <= rank <= size:\n            raise ValueError(\"every rank must be an int in 1..list size\")\n    out: dict[str, Any] = {\n        \"n_queries\": len(ranks),\n        \"candidates\": {\n            \"min\": min(list_sizes),\n            \"max\": max(list_sizes),\n            \"mean\": sum(list_sizes) / len(list_sizes),\n        },\n    }\n    for k in RECALL_AT:\n        out[f\"recall@{k}\"] = sum(1 for r in ranks if r <= k) / len(ranks)\n    out[\"mrr\"] = sum(1.0 / r for r in ranks) / len(ranks)\n    out[\"median_rank\"] = sorted(ranks)[len(ranks) // 2]\n    out[\"definitions\"] = dict(METRIC_DEFINITIONS)\n    return out\n\n\ndef random_floor(list_sizes: Sequence[int]) -> dict[str, Any]:\n    \"\"\"Expected metrics of a uniformly random ordering of every query's candidate list.\"\"\"\n    if not list_sizes or any(not isinstance(n, int) or n < 2 for n in list_sizes):\n        raise ValueError(\"every candidate list needs at least two entries\")\n    out: dict[str, Any] = {\"n_queries\": len(list_sizes)}\n    for k in RECALL_AT:\n        out[f\"recall@{k}\"] = sum(min(k, n) / n for n in list_sizes) / len(list_sizes)\n    out[\"mrr\"] = sum(sum(1.0 / r for r in range(1, n + 1)) / n for n in list_sizes) / len(list_sizes)\n    out[\"baseline\"] = \"uniformly random ordering of each candidate list (expected values)\"\n    return out\n\n\ndef _tokens(text: str) -> set[str]:\n    return set(_TOKEN_RE.findall(text.lower()))\n\n\ndef jaccard(a: str, b: str) -> float:\n    x, y = _tokens(a), _tokens(b)\n    if not x or not y:\n        return 0.0\n    return len(x & y) / len(x | y)\n\n\ndef candidate_list(record: Mapping[str, Any]) -> list[str]:\n    \"\"\"A record's candidate list: its positive first, then its negatives.\"\"\"\n    return [str(record[\"positive\"]), *(str(n) for n in record[\"negatives\"])]\n\n\ndef lexical_baseline(records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:\n    \"\"\"Order every candidate list by Jaccard token overlap with the query — the no-model floor.\"\"\"\n    ranks, sizes = [], []\n    for record in records:\n        candidates = candidate_list(record)\n        scores = [jaccard(record[\"query\"], doc) for doc in candidates]\n        ranks.append(rank_of_positive(scores, 0))\n        sizes.append(len(candidates))\n    result = ranking_metrics(ranks, sizes)\n    result[\"baseline\"] = \"Jaccard overlap of lower-cased alphanumeric tokens between query and candidate\"\n    return result\n","qwen3_reranker_pipeline/pipeline.py":"from __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nimport time\nfrom collections.abc import Callable, Mapping, Sequence\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\n\nMODEL_ID = \"Qwen/Qwen3-Reranker-0.6B\"\nMODEL_REVISION = \"e61197ed45024b0ed8a2d74b80b4d909f1255473\"\nMODEL_LICENSE = \"apache-2.0\"\nMODEL_KEY = \"qwen3-reranker-0.6b\"\nDEFAULT_WEIGHTS_DIR = Path(__file__).resolve().parents[2] / \"weights\" / MODEL_KEY\nMANIFEST_NAME = \"dimer-base-manifest.json\"\n\n# Prompt contract from the pinned upstream README (\"Using Transformers\"): a fixed system prompt, the\n# instruction/query/document block, and the assistant prefix with an empty <think> block; the score is\n# the softmax over the \"no\"/\"yes\" logits at the last position. Token ids are pinned by the snapshot's\n# 1_LogitScore/config.json and cross-checked against the tokenizer at load time.\nPREFIX = (\n    \"<|im_start|>system\\nJudge whether the Document meets the requirements based on the Query and the \"\n    'Instruct provided. Note that the answer can only be \"yes\" or \"no\".<|im_end|>\\n<|im_start|>user\\n'\n)\nSUFFIX = \"<|im_end|>\\n<|im_start|>assistant\\n<think>\\n\\n</think>\\n\\n\"\nDEFAULT_INSTRUCTION = \"Given a web search query, retrieve relevant passages that answer the query\"\nYES_TOKEN, NO_TOKEN = \"yes\", \"no\"\nYES_TOKEN_ID, NO_TOKEN_ID = 9693, 2152\nMAX_TEXT_TOKENS = 8192  # total prompt length incl. prefix/suffix; the pair is truncated longest-first to fit\nMAX_TEXT_CHARS = 100_000  # per query or document, pre-tokenisation guard\nMAX_PAIRS = 32  # (query, document) pairs per rerank() call\nSCORE_KIND = \"relevance score, not a calibrated probability\"\nWEIGHT_FILE = \"model.safetensors\"\nWEIGHT_SHA256 = (\n    \"27cd75a405b9c1b46b59abfd88aaa209e6fed2a1972cde9b70e7659537c5e65b\"  # manifest digest of WEIGHT_FILE\n)\nPARAMETER_COUNT = 595_776_512  # Qwen3ForCausalLM with the output projection tied to the token embeddings\nDECODER_LAYERS = 28  # config.json num_hidden_layers\nDEFAULT_TRAINABLE_LAYERS = 2  # the last two decoder layers (31,461,888 parameters)\nMAX_TRAIN_TOKENS = 192  # training-only prompt ceiling (inference truncates the pair to MAX_TEXT_TOKENS)\nMAX_TRAIN_CANDIDATES = 4  # per query: the positive plus the first negatives, scored together in one list\nMAX_EVAL_RECORDS = 2_000\nMIN_SCORED_RECORDS = 50  # below this a scored dataset is labelled a small sample\nARTIFACT_FORMAT = \"org.valcorza.qwen3-reranker-0.6b.adapter.v1\"\nARTIFACT_FORMAT_VERSION = \"1.0\"\nARTIFACT_WEIGHTS_NAME = \"adapter.safetensors\"\nARTIFACT_MANIFEST_NAME = \"manifest.json\"\n\n\ndef _sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with open(path, \"rb\") as handle:\n        for chunk in iter(lambda: handle.read(1 << 20), b\"\"):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:\n    \"\"\"Check the local snapshot against its DIMER manifest; raise naming the first mismatch.\"\"\"\n    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR\n    manifest_path = root / MANIFEST_NAME\n    if not manifest_path.is_file():\n        raise FileNotFoundError(f\"snapshot manifest not found: {manifest_path}\")\n    manifest = json.loads(manifest_path.read_text(encoding=\"utf-8\"))\n    if manifest.get(\"modelId\") != MODEL_ID:\n        raise ValueError(f\"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}\")\n    if manifest.get(\"revision\") != MODEL_REVISION:\n        raise ValueError(f\"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}\")\n    for entry in manifest[\"files\"]:\n        file_path = root / entry[\"path\"]\n        if not file_path.is_file():\n            raise FileNotFoundError(f\"snapshot file missing: {file_path}\")\n        size = file_path.stat().st_size\n        if size != entry[\"bytes\"]:\n            raise ValueError(f\"{entry['path']}: size {size} != manifest {entry['bytes']}\")\n        digest = hashlib.sha256()\n        with open(file_path, \"rb\") as fh:\n            for chunk in iter(lambda: fh.read(1 << 20), b\"\"):\n                digest.update(chunk)\n        if digest.hexdigest() != entry[\"sha256\"]:\n            raise ValueError(f\"{entry['path']}: sha256 {digest.hexdigest()} != manifest {entry['sha256']}\")\n    return manifest\n\n\ndef _hub_download(relative_path: str, root: Path) -> None:\n    \"\"\"Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory.\"\"\"\n    from huggingface_hub import hf_hub_download\n\n    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))\n\n\ndef stage_missing_files(\n    path: str | Path | None = None,\n    *,\n    allow_download: bool = False,\n    downloader: Callable[[str, Path], None] | None = None,\n) -> list[str]:\n    \"\"\"Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but\n    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after.\"\"\"\n    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR\n    manifest_path = root / MANIFEST_NAME\n    if not manifest_path.is_file():\n        raise FileNotFoundError(f\"manifest not found: {manifest_path}\")\n    with open(manifest_path, encoding=\"utf-8\") as fh:\n        manifest = json.load(fh)\n    if manifest.get(\"modelId\") != MODEL_ID or manifest.get(\"revision\") != MODEL_REVISION:\n        raise ValueError(\n            f\"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, \"\n            f\"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage\"\n        )\n    missing = [entry[\"path\"] for entry in manifest[\"files\"] if not (root / entry[\"path\"]).is_file()]\n    if not missing:\n        return []\n    if not allow_download:\n        raise FileNotFoundError(\n            f\"snapshot at {root} is missing {missing}; \"\n            f\"pass allow_download=True to fetch them at {MODEL_REVISION}\"\n        )\n    fetch = downloader or _hub_download\n    for relative_path in missing:\n        fetch(relative_path, root)\n    return missing\n\n\ndef format_pair(query: str, document: str, instruction: str = DEFAULT_INSTRUCTION) -> str:\n    \"\"\"Upstream `format_instruction`: the user-turn body between PREFIX and SUFFIX.\"\"\"\n    return f\"<Instruct>: {instruction}\\n<Query>: {query}\\n<Document>: {document}\"\n\n\nINPUT_SCHEMA: dict[str, Any] = {\n    \"input\": \"sequence of (query, document) pairs of two non-empty str; one score is returned per pair\",\n    \"pairs\": [1, MAX_PAIRS],\n    \"text_chars\": [1, MAX_TEXT_CHARS],\n    \"prompt_tokens\": [1, MAX_TEXT_TOKENS],\n    \"score_range\": [0.0, 1.0],\n    \"preprocessing\": (\n        \"each pair becomes '<Instruct>: <instruction>\\\\n<Query>: …\\\\n<Document>: …' between the fixed \"\n        \"upstream PREFIX and SUFFIX, truncated longest-first to fit MAX_TEXT_TOKENS; the score is the \"\n        \"two-way softmax share of the yes logit against the no logit at the last position\"\n    ),\n}\n\n\ndef _check_inputs(pairs: Any, instruction: str) -> list[tuple[str, str]]:\n    \"\"\"Raise TypeError/ValueError naming the first violated ceiling; return the pairs as a list.\"\"\"\n    if isinstance(pairs, str | bytes) or not isinstance(pairs, Sequence):\n        raise TypeError(\"pairs must be a list of (query, document) pairs\")\n    if not 1 <= len(pairs) <= MAX_PAIRS:\n        raise ValueError(f\"pairs must hold 1..{MAX_PAIRS} items, got {len(pairs)}\")\n    for i, pair in enumerate(pairs):\n        if isinstance(pair, str | bytes) or not isinstance(pair, Sequence) or len(pair) != 2:\n            raise TypeError(f\"pairs[{i}] must be a (query, document) pair of two str\")\n        for name, text in zip((\"query\", \"document\"), pair, strict=True):\n            if not isinstance(text, str):\n                raise TypeError(f\"pairs[{i}] {name} must be str, got {type(text).__name__}\")\n            if not text.strip():\n                raise ValueError(f\"pairs[{i}] {name} is empty\")\n            if len(text) > MAX_TEXT_CHARS:\n                raise ValueError(f\"pairs[{i}] {name} has {len(text)} chars; ceiling is {MAX_TEXT_CHARS}\")\n    if not isinstance(instruction, str) or not instruction.strip():\n        raise ValueError(\"instruction must be a non-empty str\")\n    return [(pair[0], pair[1]) for pair in pairs]\n\n\ndef validate_inputs(\n    pairs: Sequence[Sequence[str]],\n    instruction: str = DEFAULT_INSTRUCTION,\n    *,\n    names: Sequence[str] | None = None,\n) -> dict[str, Any]:\n    \"\"\"Validation stage: return the input manifest (schema, per-pair observations, verdict).\n\n    Rejection is reported by raising exactly as ``rerank`` would — both route through\n    ``_check_inputs``. Prompt-level truncation cannot be observed here because it happens inside\n    the tokenizer; ``rerank`` reports it in ``truncated``.\n    \"\"\"\n    checked = _check_inputs(pairs, instruction)\n    if names is not None and len(names) != len(checked):\n        raise ValueError(\"names must have one entry per pair\")\n    return {\n        \"schema\": dict(INPUT_SCHEMA),\n        \"inputs\": [\n            {\n                \"id\": names[i] if names else f\"pair-{i}\",\n                \"query_chars\": len(query),\n                \"document_chars\": len(document),\n            }\n            for i, (query, document) in enumerate(checked)\n        ],\n        \"n_pairs\": len(checked),\n        \"instruction\": instruction,\n        \"verdict\": \"accepted\",\n        \"findings\": [],\n        \"model_id\": MODEL_ID,\n        \"model_revision\": MODEL_REVISION,\n    }\n\n\ndef evaluation_report(\n    result: Mapping[str, Any], judgements: Sequence[Any] | None = None, *, sample_kind: str = \"synthetic\"\n) -> dict[str, Any]:\n    \"\"\"Evaluation stage: a machine-readable report even though no metric exists here.\n\n    The repository ships no ranking-metric helper, so the verdict is always ``not-measurable``\n    (EVAL9), including when ``judgements`` is supplied: the parameter exists for interface parity\n    with the fleet's other pipelines and is recorded in ``reason`` rather than scored. Inventing\n    nDCG or MRR here would hide the fact that a real evaluation needs judged candidates over many\n    queries and the caller's own metric code.\n    \"\"\"\n    scores = result[\"scores\"]\n    supplied = judgements is not None\n    return {\n        \"task\": \"pointwise query-document relevance reranking\",\n        \"score_semantics\": (\n            f\"{SCORE_KIND}: the softmax share of the yes logit against the no logit, in [0, 1]; it \"\n            \"orders candidates for one query, is not comparable as an absolute value across queries \"\n            \"or instructions, and carries no shipped acceptance threshold\"\n        ),\n        \"sample_kind\": sample_kind,\n        \"n_pairs\": len(scores),\n        \"metrics\": [],\n        \"baselines\": [],\n        \"verdict\": \"not-measurable\",\n        \"reason\": (\n            \"ranking quality needs relevance judgements and the repository ships no metric helper\"\n            + (\n                \"; judgements were supplied but no metric helper exists to score them here\"\n                if supplied\n                else \"; the evaluated sample carries none\"\n            )\n        ),\n        \"needs\": (\n            \"per-query relevance judgements (binary or graded) over enough queries to state a \"\n            \"dispersion, scored with the caller's own nDCG@k, MRR or precision@k code; a single \"\n            \"query's ordering is a plumbing check, not a retrieval measurement\"\n        ),\n        \"model_id\": MODEL_ID,\n        \"model_revision\": MODEL_REVISION,\n    }\n\n\n@dataclass\nclass Qwen3RerankerPipeline:\n    \"\"\"Pointwise reranker. `_runner` maps pair bodies to ([no, yes] last-position logits, token counts).\"\"\"\n\n    _runner: Callable[[list[str]], tuple[np.ndarray, list[int]]]\n    device: str\n    adapter: dict[str, Any] | None = field(default=None, repr=False)\n    _model: Any = field(default=None, repr=False)\n    _tokenizer: Any = field(default=None, repr=False)\n\n    @classmethod\n    def from_pretrained(\n        cls,\n        device: str | None = None,\n        weights_dir: str | Path | None = None,\n        allow_download: bool = False,\n    ) -> Qwen3RerankerPipeline:\n        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR\n        if (root / MANIFEST_NAME).is_file():\n            stage_missing_files(root, allow_download=allow_download)\n            verify_snapshot(root)\n            source, kwargs = str(root), dict(local_files_only=True)\n        elif allow_download:\n            source, kwargs = MODEL_ID, dict(revision=MODEL_REVISION)\n        else:\n            raise FileNotFoundError(f\"no verified snapshot at {root} and allow_download=False\")\n        # Refuse invalid snapshots before importing model libraries.\n        import torch\n        from transformers import AutoModelForCausalLM, AutoTokenizer\n\n        resolved_device = device or (\"cuda:0\" if torch.cuda.is_available() else \"cpu\")\n        dtype = torch.bfloat16 if resolved_device.startswith(\"cuda\") else torch.float32\n        tokenizer = AutoTokenizer.from_pretrained(\n            source, padding_side=\"left\", trust_remote_code=False, **kwargs\n        )\n        ids = (tokenizer.convert_tokens_to_ids(YES_TOKEN), tokenizer.convert_tokens_to_ids(NO_TOKEN))\n        if ids != (YES_TOKEN_ID, NO_TOKEN_ID):\n            raise RuntimeError(f\"tokenizer maps yes/no to {ids}, expected {(YES_TOKEN_ID, NO_TOKEN_ID)}\")\n        model = AutoModelForCausalLM.from_pretrained(source, dtype=dtype, trust_remote_code=False, **kwargs)\n        model = model.to(resolved_device).eval()\n        prefix_ids = tokenizer.encode(PREFIX, add_special_tokens=False)\n        suffix_ids = tokenizer.encode(SUFFIX, add_special_tokens=False)\n        body_budget = MAX_TEXT_TOKENS - len(prefix_ids) - len(suffix_ids)\n\n        def runner(bodies: list[str]) -> tuple[np.ndarray, list[int]]:\n            enc = tokenizer(\n                bodies,\n                padding=False,\n                truncation=\"longest_first\",\n                return_attention_mask=False,\n                max_length=body_budget,\n            )\n            enc[\"input_ids\"] = [prefix_ids + row + suffix_ids for row in enc[\"input_ids\"]]\n            batch = tokenizer.pad(enc, padding=True, return_tensors=\"pt\")\n            batch = batch.to(resolved_device)\n            with torch.inference_mode():\n                last = model(**batch).logits[:, -1, :]\n            pair_logits = torch.stack([last[:, NO_TOKEN_ID], last[:, YES_TOKEN_ID]], dim=1)\n            counts = batch[\"attention_mask\"].sum(dim=1).tolist()\n            return pair_logits.float().cpu().numpy(), [int(c) for c in counts]\n\n        return cls(runner, resolved_device, _model=model, _tokenizer=tokenizer)\n\n    def _validate(self, pairs: Any, instruction: str) -> list[tuple[str, str]]:\n        return _check_inputs(pairs, instruction)\n\n    def rerank(\n        self,\n        pairs: Sequence[Sequence[str]],\n        instruction: str = DEFAULT_INSTRUCTION,\n    ) -> dict[str, Any]:\n        \"\"\"Score up to MAX_PAIRS (query, document) pairs; `scores` align with `pairs`; no threshold.\"\"\"\n        pairs = self._validate(pairs, instruction)\n        bodies = [format_pair(q, d, instruction) for q, d in pairs]\n        logits, n_tokens = self._runner(bodies)\n        logits = np.asarray(logits, dtype=np.float64)\n        if logits.shape != (len(pairs), 2):\n            raise RuntimeError(f\"backend returned {logits.shape}, expected ({len(pairs)}, 2)\")\n        shifted = logits - logits.max(axis=1, keepdims=True)\n        probs = np.exp(shifted) / np.exp(shifted).sum(axis=1, keepdims=True)\n        scores = probs[:, 1]\n        return {\n            \"scores\": [float(s) for s in scores],\n            \"ranking\": [int(i) for i in np.argsort(-scores, kind=\"stable\")],\n            \"score_kind\": SCORE_KIND,\n            \"instruction\": instruction,\n            \"n_tokens\": list(n_tokens),\n            \"truncated\": [n >= MAX_TEXT_TOKENS for n in n_tokens],\n            \"model_id\": MODEL_ID,\n            \"model_revision\": MODEL_REVISION,\n        }\n\n    # ---- adaptation -----------------------------------------------------------------------------------\n\n    def _require_model(self) -> tuple[Any, Any]:\n        if self._model is None or self._tokenizer is None:\n            raise ValueError(\n                \"this operation needs a pipeline built with from_pretrained() or from_artifact()\"\n            )\n        return self._model, self._tokenizer\n\n    def _score_pairs(self, pairs: Sequence[tuple[str, str]], instruction: str) -> list[float]:\n        \"\"\"Score any number of pairs through the public contract, MAX_PAIRS at a time.\"\"\"\n        scores: list[float] = []\n        for start in range(0, len(pairs), MAX_PAIRS):\n            scores.extend(self.rerank(list(pairs[start : start + MAX_PAIRS]), instruction)[\"scores\"])\n        return scores\n\n    def evaluate(\n        self, records: Sequence[Mapping[str, Any]], *, instruction: str = DEFAULT_INSTRUCTION\n    ) -> dict[str, Any]:\n        \"\"\"Rerank every record's candidate list (its positive followed by its negatives) with `rerank` and\n        read the rank of the positive: recall@1 / recall@3 / recall@5 and MRR, ties against the positive.\"\"\"\n        from .metrics import candidate_list, rank_of_positive, ranking_metrics\n        from .samples import validate_dataset\n\n        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)[\"records\"]\n        _check_inputs([(\"x\", \"y\")], instruction)\n        started = time.perf_counter()\n        pairs: list[tuple[str, str]] = []\n        sizes = []\n        for record in checked:\n            candidates = candidate_list(record)\n            sizes.append(len(candidates))\n            pairs.extend((record[\"query\"], doc) for doc in candidates)\n        scores = self._score_pairs(pairs, instruction)\n        ranks = []\n        cursor = 0\n        for size in sizes:\n            ranks.append(rank_of_positive(scores[cursor : cursor + size], 0))\n            cursor += size\n        metrics = ranking_metrics(ranks, sizes)\n        metrics.update(\n            {\n                \"instruction\": instruction,\n                \"n_pairs\": len(pairs),\n                \"verdict\": \"measured\" if len(checked) >= MIN_SCORED_RECORDS else \"measured-small-sample\",\n                \"adapted\": self.adapter is not None,\n                \"seconds\": round(time.perf_counter() - started, 3),\n                \"model_id\": MODEL_ID,\n                \"model_revision\": MODEL_REVISION,\n            }\n        )\n        return metrics\n\n    @staticmethod\n    def lexical_baseline(records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:\n        \"\"\"The no-model floor: candidate lists ordered by token overlap with the query (see metrics.py).\"\"\"\n        from .metrics import lexical_baseline\n        from .samples import validate_dataset\n\n        return lexical_baseline(\n            validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)[\"records\"]\n        )\n\n    def _trainable_names(self, trainable_layers: int) -> list[str]:\n        if not isinstance(trainable_layers, int) or not 1 <= trainable_layers <= DECODER_LAYERS:\n            raise ValueError(f\"trainable_layers must be an int in 1..{DECODER_LAYERS}\")\n        model, _ = self._require_model()\n        first = DECODER_LAYERS - trainable_layers\n        prefixes = tuple(f\"model.layers.{k}.\" for k in range(first, DECODER_LAYERS))\n        return [name for name, _p in model.named_parameters() if name.startswith(prefixes)]\n\n    def adapt(\n        self,\n        train: Sequence[Mapping[str, Any]],\n        val: Sequence[Mapping[str, Any]] | None = None,\n        *,\n        instruction: str = DEFAULT_INSTRUCTION,\n        epochs: int = 2,\n        lr: float = 2e-5,\n        batch_size: int = 4,\n        trainable_layers: int = DEFAULT_TRAINABLE_LAYERS,\n        train_candidates: int = MAX_TRAIN_CANDIDATES,\n        seed: int = 0,\n        progress: Callable[[dict[str, Any]], None] | None = None,\n    ) -> dict[str, Any]:\n        \"\"\"Bounded listwise fine-tuning on validated query / positive / negatives records.\n\n        Only the last `trainable_layers` decoder layers train (2 by default; the token embeddings — which the\n        output projection shares — the earlier layers and the final norm stay frozen). Each query contributes\n        one list: its positive and its first `train_candidates - 1` negatives, formatted exactly as `rerank`\n        formats them; the relevance logit of every candidate is the yes-minus-no logit at the last position,\n        and the loss is the cross-entropy of the positive over its list (listwise softmax). `batch_size`\n        counts queries per step; AdamW at a fixed learning rate, gradient clipping at 1.0, seeded shuffling,\n        no scheduler; prompts are truncated to MAX_TRAIN_TOKENS **during training only**. Epoch 0 records the\n        frozen model's validation ranking metrics; the epoch with the highest validation MRR is kept.\"\"\"\n        from .metrics import candidate_list\n        from .samples import validate_dataset\n\n        if not isinstance(epochs, int) or not 1 <= epochs <= 20:\n            raise ValueError(\"epochs must be an int in 1..20\")\n        if not (0.0 < lr <= 1e-3):\n            raise ValueError(\"lr must be in (0, 1e-3]\")\n        if not isinstance(batch_size, int) or not 1 <= batch_size <= 16:\n            raise ValueError(\"batch_size must be an int in 1..16\")\n        if not isinstance(train_candidates, int) or not 2 <= train_candidates <= 8:\n            raise ValueError(\"train_candidates must be an int in 2..8\")\n        _check_inputs([(\"x\", \"y\")], instruction)\n        names = self._trainable_names(trainable_layers)\n        train_checked = validate_dataset(train)[\"records\"]\n        val_checked = (\n            validate_dataset(val, min_records=1, max_records=MAX_EVAL_RECORDS)[\"records\"] if val else []\n        )\n        import torch\n\n        torch.manual_seed(seed)\n        model, tokenizer = self._require_model()\n        started = time.perf_counter()\n        wanted = set(names)\n        for name, param in model.named_parameters():\n            param.requires_grad_(name in wanted)\n        params = [p for p in model.parameters() if p.requires_grad]\n        n_trainable = sum(p.numel() for p in params)\n        optimiser = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)\n        device = torch.device(self.device)\n        prefix_ids = tokenizer.encode(PREFIX, add_special_tokens=False)\n        suffix_ids = tokenizer.encode(SUFFIX, add_special_tokens=False)\n        body_budget = MAX_TRAIN_TOKENS - len(prefix_ids) - len(suffix_ids)\n\n        def score_val() -> dict[str, Any] | None:\n            if not val_checked:\n                return None\n            model.eval()\n            keep = (\"recall@1\", \"recall@3\", \"recall@5\", \"mrr\", \"n_pairs\")\n            return {k: v for k, v in self.evaluate(val_checked, instruction=instruction).items() if k in keep}\n\n        def relevance(bodies: list[str]) -> torch.Tensor:\n            enc = tokenizer(\n                bodies,\n                padding=False,\n                truncation=\"longest_first\",\n                max_length=body_budget,\n                return_attention_mask=False,\n            )\n            enc[\"input_ids\"] = [prefix_ids + row + suffix_ids for row in enc[\"input_ids\"]]\n            batch = tokenizer.pad(enc, padding=True, return_tensors=\"pt\").to(device)\n            last = model(**batch).logits[:, -1, :].float()\n            return last[:, YES_TOKEN_ID] - last[:, NO_TOKEN_ID]\n\n        history: list[dict[str, Any]] = []\n        entry: dict[str, Any] = {\"epoch\": 0, \"train_loss\": None, \"val\": score_val(), \"note\": \"frozen model\"}\n        history.append(entry)\n        if progress:\n            progress(entry)\n        best_mrr = entry[\"val\"][\"mrr\"] if entry[\"val\"] else -math.inf\n        best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}\n        initial_state = {k: v.clone() for k, v in best_state.items()}\n        best_epoch = 0\n        generator = torch.Generator().manual_seed(seed)\n        lists = [candidate_list(r)[:train_candidates] for r in train_checked]\n        try:\n            for epoch in range(1, epochs + 1):\n                model.train()\n                order = torch.randperm(len(train_checked), generator=generator).tolist()\n                losses = []\n                for start in range(0, len(order), batch_size):\n                    chosen = order[start : start + batch_size]\n                    bodies, spans = [], []\n                    for i in chosen:\n                        spans.append((len(bodies), len(lists[i])))\n                        bodies.extend(\n                            format_pair(train_checked[i][\"query\"], doc, instruction) for doc in lists[i]\n                        )\n                    logits = relevance(bodies)\n                    loss = torch.stack(\n                        [\n                            torch.nn.functional.cross_entropy(\n                                logits[a : a + n][None], torch.zeros(1, dtype=torch.long, device=device)\n                            )\n                            for a, n in spans\n                        ]\n                    ).mean()\n                    optimiser.zero_grad(set_to_none=True)\n                    loss.backward()\n                    torch.nn.utils.clip_grad_norm_(params, 1.0)\n                    optimiser.step()\n                    losses.append(float(loss.detach()))\n                model.eval()\n                entry = {\"epoch\": epoch, \"train_loss\": sum(losses) / max(len(losses), 1), \"val\": score_val()}\n                history.append(entry)\n                if progress:\n                    progress(entry)\n                current = entry[\"val\"][\"mrr\"] if entry[\"val\"] else math.inf\n                if current > best_mrr or not entry[\"val\"]:\n                    best_mrr = current\n                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}\n                    best_epoch = epoch\n        except BaseException:\n            # Transactional: a failure in training, validation or the progress callback leaves the base\n            # exactly as it was, with every parameter frozen again.\n            restore = dict(model.state_dict())\n            restore.update(initial_state)\n            model.load_state_dict(restore, strict=True)\n            model.eval()\n            for param in model.parameters():\n                param.requires_grad_(False)\n            self.adapter = None\n            raise\n        merged = dict(model.state_dict())\n        merged.update(best_state)\n        model.load_state_dict(merged, strict=True)\n        model.eval()\n        for param in model.parameters():\n            param.requires_grad_(False)\n        self.adapter = {\n            \"objective\": \"listwise cross-entropy of the positive over its candidates (yes-minus-no logit)\",\n            \"instruction\": instruction,\n            \"trainable_layers\": trainable_layers,\n            \"trainable_names\": names,\n            \"n_trainable\": n_trainable,\n            \"n_total\": sum(p.numel() for p in model.parameters()),\n            \"epochs\": epochs,\n            \"best_epoch\": best_epoch,\n            \"selection\": \"highest validation MRR\" if val_checked else \"final epoch (no validation split)\",\n            \"lr\": lr,\n            \"batch_size\": batch_size,\n            \"train_candidates\": train_candidates,\n            \"max_train_tokens\": MAX_TRAIN_TOKENS,\n            \"n_train\": len(train_checked),\n            \"n_train_pairs\": sum(len(lst) for lst in lists),\n            \"n_val\": len(val_checked),\n            \"seed\": seed,\n            \"history\": history,\n            \"seconds\": round(time.perf_counter() - started, 2),\n        }\n        return dict(self.adapter)\n\n    # ---- artifacts ------------------------------------------------------------------------------------\n\n    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:\n        \"\"\"Write the adapted decoder-layer tensors as safetensors with a manifest naming the base.\"\"\"\n        if self.adapter is None:\n            raise ValueError(\"nothing to save: call adapt() first\")\n        model, _ = self._require_model()\n        from safetensors.torch import save_file\n\n        out = Path(output_dir)\n        out.mkdir(parents=True, exist_ok=True)\n        names = set(self.adapter[\"trainable_names\"])\n        tensors = {k: v.detach().cpu().contiguous() for k, v in model.state_dict().items() if k in names}\n        weights_path = out / ARTIFACT_WEIGHTS_NAME\n        save_file(tensors, str(weights_path), metadata={\"format\": \"pt\"})\n        manifest = {\n            \"format\": ARTIFACT_FORMAT,\n            \"format_version\": ARTIFACT_FORMAT_VERSION,\n            \"base_model\": {\n                \"id\": MODEL_ID,\n                \"revision\": MODEL_REVISION,\n                \"key\": MODEL_KEY,\n                \"weight_file\": WEIGHT_FILE,\n                \"weight_sha256\": WEIGHT_SHA256,\n            },\n            \"adapter\": {k: v for k, v in self.adapter.items() if k not in (\"history\", \"trainable_names\")},\n            \"history\": self.adapter[\"history\"],\n            \"tensors\": sorted(tensors),\n            \"files\": [\n                {\n                    \"path\": ARTIFACT_WEIGHTS_NAME,\n                    \"bytes\": weights_path.stat().st_size,\n                    \"sha256\": _sha256(weights_path),\n                }\n            ],\n            \"metadata\": dict(metadata or {}),\n        }\n        (out / ARTIFACT_MANIFEST_NAME).write_text(\n            json.dumps(manifest, indent=2, ensure_ascii=False), encoding=\"utf-8\"\n        )\n        return out\n\n    def _check_artifact_manifest(self, root: Path, manifest: Mapping[str, Any]) -> Path:\n        \"\"\"Refuse an artifact whose manifest is not exactly the one this pipeline writes: the supported\n        format and version, the pinned base (id, revision, weight file, digest), exactly one file entry\n        named `adapter.safetensors` that resolves inside the artifact directory, and a recorded\n        `trainable_layers` in range. Nothing is deserialised here. The digest check that follows\n        detects corruption or drift of the weights relative to the adjacent manifest; it is not\n        authenticity against an actor who can replace both files.\"\"\"\n        if manifest.get(\"format\") != ARTIFACT_FORMAT:\n            raise ValueError(f\"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}\")\n        if manifest.get(\"format_version\") != ARTIFACT_FORMAT_VERSION:\n            raise ValueError(\n                f\"artifact format_version {manifest.get('format_version')!r} is not the supported \"\n                f\"{ARTIFACT_FORMAT_VERSION!r}\"\n            )\n        base = manifest.get(\"base_model\", {})\n        if (base.get(\"id\"), base.get(\"revision\"), base.get(\"weight_sha256\")) != (\n            MODEL_ID,\n            MODEL_REVISION,\n            WEIGHT_SHA256,\n        ):\n            raise ValueError(\"artifact was adapted from a different base model, revision or weight file\")\n        if base.get(\"weight_file\", WEIGHT_FILE) != WEIGHT_FILE:\n            raise ValueError(\"artifact was adapted from a different base weight file\")\n        files = manifest.get(\"files\")\n        if not isinstance(files, list) or len(files) != 1:\n            raise ValueError(\"artifact manifest must list exactly one file\")\n        entry = files[0]\n        if not isinstance(entry, Mapping) or entry.get(\"path\") != ARTIFACT_WEIGHTS_NAME:\n            raise ValueError(f\"artifact manifest must name exactly {ARTIFACT_WEIGHTS_NAME!r}\")\n        weights_path = (root / entry[\"path\"]).resolve()\n        if weights_path.parent != root.resolve():\n            raise ValueError(\"artifact weight path must resolve inside the artifact directory\")\n        adapter = manifest.get(\"adapter\")\n        layers = adapter.get(\"trainable_layers\") if isinstance(adapter, Mapping) else None\n        if isinstance(layers, bool) or not isinstance(layers, int):\n            raise ValueError(\"artifact manifest does not record an integer trainable_layers\")\n        if not isinstance(manifest.get(\"tensors\"), list):\n            raise ValueError(\"artifact manifest must list its tensors\")\n        return weights_path\n\n    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:\n        \"\"\"Verify an adapter's manifest, digest and exact tensor set **before** deserialising, then overwrite\n        exactly the tensors it carries.\"\"\"\n        root = Path(artifact_dir)\n        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding=\"utf-8\"))\n        weights_path = self._check_artifact_manifest(root, manifest)\n        entry = manifest[\"files\"][0]\n        if not weights_path.is_file():\n            raise FileNotFoundError(f\"artifact weights missing: {weights_path}\")\n        if _sha256(weights_path) != entry[\"sha256\"] or weights_path.stat().st_size != entry[\"bytes\"]:\n            raise ValueError(f\"{entry['path']}: digest or size mismatch; refusing to load\")\n        # The exact tensor set the recorded configuration implies — no subset, no extra, no other layer.\n        expected = sorted(self._trainable_names(manifest[\"adapter\"][\"trainable_layers\"]))\n        if sorted(manifest[\"tensors\"]) != expected:\n            raise ValueError(\"artifact tensor list does not match its recorded configuration\")\n        model, _ = self._require_model()\n        from safetensors.torch import load_file\n\n        tensors = load_file(str(weights_path))\n        if sorted(tensors) != expected:\n            raise ValueError(\"artifact tensor names differ from its manifest\")\n        state = model.state_dict()\n        for key, value in tensors.items():\n            if key not in state or not key.startswith(\"model.layers.\"):\n                raise ValueError(\n                    f\"artifact tensor {key} is not an adaptable decoder-layer tensor of the base\"\n                )\n            if tuple(value.shape) != tuple(state[key].shape):\n                raise ValueError(\n                    f\"artifact tensor {key}: shape {tuple(value.shape)} != {tuple(state[key].shape)}\"\n                )\n        merged = dict(state)\n        merged.update({k: v.to(state[k].dtype) for k, v in tensors.items()})\n        model.load_state_dict(merged, strict=True)\n        model.eval()\n        self.adapter = {\n            **manifest[\"adapter\"],\n            \"trainable_names\": manifest[\"tensors\"],\n            \"history\": manifest.get(\"history\", []),\n        }\n        return manifest\n\n    @classmethod\n    def from_artifact(\n        cls,\n        artifact_dir: str | Path,\n        *,\n        device: str | None = None,\n        weights_dir: str | Path | None = None,\n        allow_download: bool = False,\n    ) -> Qwen3RerankerPipeline:\n        pipeline = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)\n        pipeline.load_artifact(artifact_dir)\n        return pipeline\n","qwen3_reranker_pipeline/samples.py":"\"\"\"Query / positive / negatives dataset contract for reranker fine-tuning: the pinned Banking77 sample,\nvalidation, seeded splitting, BYOD loaders and CSV export.\n\nThe default dataset is **real** and a reranking task the model was not tuned for: Banking77 (Casanueva et\nal., 2020; CC BY 4.0), 13,083 customer-support messages labelled with 77 fine-grained banking intents. Two\nCSV files (`train.csv`, `test.csv`) are fetched from the PolyAI `task-specific-datasets` repository at a\npinned commit and refused on any byte-size or SHA-256 mismatch. Every intent name becomes a short\n**document** (`card_arrival` → `card arrival`); each message is a **query** whose positive document is its\nintent phrase, and its **negatives** are a seeded shortlist of other intent phrases — the hardest few by\ntoken overlap with the message plus random ones — so reranking the shortlist is intent detection over a\ncandidate list. Training and validation queries are drawn from `train.csv`, test queries from `test.csv` —\nthe release's own partition — balanced over the 77 intents.\n\nA record is ``{id, query, positive, negatives}``; its candidate list is the positive then the negatives.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport io\nimport json\nimport random\nimport re\nimport urllib.request\nfrom collections.abc import Mapping, Sequence\nfrom pathlib import Path\nfrom typing import Any\n\nfrom .metrics import jaccard, record_seed\nfrom .pipeline import MAX_TEXT_CHARS, MODEL_ID\n\nCORPUS_NAME = \"Banking77 (messages → intent-phrase shortlists)\"\nCORPUS_RELEASE = \"PolyAI-LDN/task-specific-datasets @ 57ec275d8078af65b7731c2a98be812d844a6d6b\"\nCORPUS_BASE_URL = (\n    \"https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/\"\n    \"57ec275d8078af65b7731c2a98be812d844a6d6b/banking_data/\"\n)\nCORPUS_FILES = {\n    \"train\": (\"train.csv\", 839_073, \"b06e26ac675513959a63135f11b94ea7786ed02da65db93a5650d8838cbc664b\"),\n    \"test\": (\"test.csv\", 239_961, \"d12d6e3bc4c3103966ae786dc435913c0c563dfa328f5a3646d0e62cfeeb474d\"),\n}\nCORPUS_LICENSE = \"CC BY 4.0 (Casanueva et al. 2020; PolyAI-LDN/task-specific-datasets)\"\nCORPUS_ROWS = {\"train\": 10_003, \"test\": 3_080}\nCORPUS_INTENTS = 77\nDEFAULT_CACHE_DIR = Path(\"weights\") / \"banking77\"\nTASK_INSTRUCTION = (\n    \"Given a customer support message, judge whether the document names the banking intent it expresses\"\n)\nSAMPLE_SEED = 42\nSAMPLE_SPLIT = {\"train\": 231, \"validation\": 77, \"test\": 154}  # 3 / 1 / 2 per intent, balanced over 77\nSAMPLE_NEGATIVES = 5  # per query: SAMPLE_HARD_NEGATIVES highest-overlap other intents plus seeded random ones\nSAMPLE_HARD_NEGATIVES = 3\nMAX_NEGATIVES = 15\nNEGATIVE_SEPARATOR = \" | \"  # the CSV column format of `negatives`\nMIN_RECORDS = 8\nMAX_RECORDS = 20_000\nMAX_DOCUMENT_CHARS = 1_000\n_ID_RE = re.compile(r\"^[A-Za-z0-9_.:-]{1,64}$\")\n\n\ndef _sha256_bytes(data: bytes) -> str:\n    return hashlib.sha256(data).hexdigest()\n\n\ndef intent_phrase(intent: str) -> str:\n    \"\"\"The document text of an intent: its snake_case name as words (`card_arrival` → `card arrival`).\"\"\"\n    return \" \".join(intent.strip().split(\"_\"))\n\n\ndef fetch_corpus(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> dict[str, bytes]:\n    \"\"\"Return the two pinned Banking77 CSVs (bytes) from the cache or the project repository, verified.\"\"\"\n    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR\n    cache.mkdir(parents=True, exist_ok=True)\n    out = {}\n    for split, (name, size, digest) in CORPUS_FILES.items():\n        local = cache / name\n        data = local.read_bytes() if local.is_file() else b\"\"\n        if len(data) != size or _sha256_bytes(data) != digest:\n            url = CORPUS_BASE_URL + name\n            if fetcher is not None:\n                data = fetcher(url)\n            else:\n                with urllib.request.urlopen(url, timeout=120) as response:  # noqa: S310 (pinned https URL)\n                    data = response.read()\n            if len(data) != size or _sha256_bytes(data) != digest:\n                raise ValueError(\n                    f\"{name}: fetched {len(data)} bytes with sha256 {_sha256_bytes(data)[:16]}…, \"\n                    f\"pinned {size} / {digest[:16]}…\"\n                )\n            local.write_bytes(data)\n        out[split] = data\n    return out\n\n\ndef read_corpus(files: Mapping[str, bytes]) -> dict[str, list[dict[str, Any]]]:\n    \"\"\"Parse the CSV members (columns `text`, `category`) into flat records keeping the raw intent name.\"\"\"\n    out = {}\n    for split in CORPUS_FILES:\n        if split not in files:\n            raise ValueError(f\"corpus is missing the {split} file\")\n        rows = list(csv.DictReader(io.StringIO(files[split].decode(\"utf-8\"))))\n        if not rows or {\"text\", \"category\"} - set(rows[0]):\n            raise ValueError(f\"{split}: expected columns text and category\")\n        if len(rows) != CORPUS_ROWS[split]:\n            raise ValueError(f\"{split}: {len(rows)} rows, expected {CORPUS_ROWS[split]}\")\n        out[split] = [\n            {\"id\": f\"{split}-{i:05d}\", \"text\": r[\"text\"].strip(), \"intent\": r[\"category\"].strip()}\n            for i, r in enumerate(rows)\n        ]\n        intents = {r[\"intent\"] for r in out[split]}\n        if len(intents) != CORPUS_INTENTS:\n            raise ValueError(f\"{split}: {len(intents)} intents, expected {CORPUS_INTENTS}\")\n    return out\n\n\ndef filter_records(records: Sequence[Mapping[str, Any]]) -> list[dict[str, Any]]:\n    \"\"\"Turn corpus rows into query–positive pairs; drop empty, over-long and repeated queries.\"\"\"\n    seen: set[str] = set()\n    kept = []\n    for record in records:\n        query = str(record[\"text\"]).strip()\n        key = query.lower()\n        if not query or key in seen or len(query) > MAX_TEXT_CHARS:\n            continue\n        seen.add(key)\n        intent = str(record[\"intent\"])\n        kept.append({\"id\": record[\"id\"], \"query\": query, \"positive\": intent_phrase(intent), \"intent\": intent})\n    return kept\n\n\ndef sample_negatives(\n    query: str,\n    positive: str,\n    phrases: Sequence[str],\n    *,\n    seed: int,\n    n_negatives: int = SAMPLE_NEGATIVES,\n    n_hard: int = SAMPLE_HARD_NEGATIVES,\n) -> list[str]:\n    \"\"\"A seeded shortlist of other phrases: the `n_hard` with the highest token overlap with the query (ties\n    broken by phrase order), then random others up to `n_negatives`.\"\"\"\n    others = [p for p in phrases if p != positive]\n    if len(others) < n_negatives:\n        raise ValueError(f\"need {n_negatives} other phrases, have {len(others)}\")\n    hard = sorted(others, key=lambda p: (-jaccard(query, p), p))[:n_hard]\n    rest = [p for p in others if p not in hard]\n    random.Random(seed).shuffle(rest)\n    return hard + rest[: n_negatives - len(hard)]\n\n\ndef build_sample_dataset(\n    corpus: Mapping[str, Sequence[Mapping[str, Any]]],\n    *,\n    seed: int = SAMPLE_SEED,\n    sizes: Mapping[str, int] | None = None,\n) -> dict[str, list[dict[str, Any]]]:\n    \"\"\"Balanced seeded draws over all 77 intents with a seeded shortlist of negatives per query: training and\n    validation from `train` (disjoint queries), test from `test`.\"\"\"\n    sizes = dict(sizes or SAMPLE_SPLIT)\n    for name, size in sizes.items():\n        if size % CORPUS_INTENTS:\n            raise ValueError(f\"{name} size {size} is not a multiple of the {CORPUS_INTENTS} intents\")\n    rng = random.Random(seed)\n    pools = {\"train\": filter_records(corpus[\"train\"]), \"test\": filter_records(corpus[\"test\"])}\n    intents = sorted({r[\"intent\"] for r in pools[\"train\"]})\n    phrases = [intent_phrase(intent) for intent in intents]\n    by_intent = {\n        split: {intent: [r for r in pool if r[\"intent\"] == intent] for intent in intents}\n        for split, pool in pools.items()\n    }\n    for split in by_intent.values():\n        for records in split.values():\n            rng.shuffle(records)\n    cursor = dict.fromkeys(intents, 0)\n    out: dict[str, list[dict[str, Any]]] = {}\n    for name, size in sizes.items():\n        source = \"test\" if name == \"test\" else \"train\"\n        per_intent = size // CORPUS_INTENTS\n        picked = []\n        for intent in intents:\n            pool = by_intent[source][intent]\n            start = cursor[intent] if source == \"train\" else 0\n            chunk = pool[start : start + per_intent]\n            if len(chunk) < per_intent:\n                raise ValueError(\n                    f\"{name}: only {len(chunk)} records available for {intent!r}, need {per_intent}\"\n                )\n            picked.extend(chunk)\n            if source == \"train\":\n                cursor[intent] = start + per_intent\n        rng.shuffle(picked)\n        out[name] = []\n        for i, r in enumerate(picked):\n            rid = f\"{name}-{i:04d}\"\n            out[name].append(\n                {\n                    \"id\": rid,\n                    \"query\": r[\"query\"],\n                    \"positive\": r[\"positive\"],\n                    \"negatives\": sample_negatives(\n                        r[\"query\"], r[\"positive\"], phrases, seed=record_seed(rid, seed)\n                    ),\n                    \"intent\": r[\"intent\"],\n                }\n            )\n    return out\n\n\ndef fetch_sample_dataset(\n    *,\n    cache_dir: str | Path | None = None,\n    fetcher: Any = None,\n    seed: int = SAMPLE_SEED,\n    sizes: Mapping[str, int] | None = None,\n) -> dict[str, list[dict[str, Any]]]:\n    \"\"\"The tutorial splits from the pinned corpus.\"\"\"\n    corpus = read_corpus(fetch_corpus(cache_dir=cache_dir, fetcher=fetcher))\n    return build_sample_dataset(corpus, seed=seed, sizes=sizes)\n\n\ndef _check_text(value: Any, label: str, ceiling: int) -> str:\n    if not isinstance(value, str):\n        raise ValueError(f\"{label} must be a string\")\n    if not value.strip():\n        raise ValueError(f\"{label} is empty\")\n    if len(value) > ceiling:\n        raise ValueError(f\"{label} has {len(value)} chars; ceiling is {ceiling}\")\n    return value.strip()\n\n\ndef _check_record(record: Any, index: int) -> dict[str, Any]:\n    label = f\"records[{index}]\"\n    if not isinstance(record, Mapping):\n        raise ValueError(f\"{label} must be a mapping with id/query/positive/negatives\")\n    for key in (\"id\", \"query\", \"positive\", \"negatives\"):\n        if key not in record:\n            raise ValueError(f\"{label} is missing {key!r}\")\n    rid = record[\"id\"]\n    if not isinstance(rid, str) or not _ID_RE.match(rid):\n        raise ValueError(f\"{label}: id must match {_ID_RE.pattern}\")\n    item = {\n        \"id\": rid,\n        \"query\": _check_text(record[\"query\"], f\"{label}: query\", MAX_TEXT_CHARS),\n        \"positive\": _check_text(record[\"positive\"], f\"{label}: positive\", MAX_DOCUMENT_CHARS),\n    }\n    negatives = record[\"negatives\"]\n    if isinstance(negatives, str | bytes) or not isinstance(negatives, Sequence):\n        raise ValueError(f\"{label}: negatives must be a list of 1..{MAX_NEGATIVES} documents\")\n    if not 1 <= len(negatives) <= MAX_NEGATIVES:\n        raise ValueError(f\"{label}: negatives must hold 1..{MAX_NEGATIVES} documents, got {len(negatives)}\")\n    checked = [\n        _check_text(n, f\"{label}: negatives[{j}]\", MAX_DOCUMENT_CHARS) for j, n in enumerate(negatives)\n    ]\n    if len(set(checked)) != len(checked):\n        raise ValueError(f\"{label}: negatives repeat a document\")\n    if item[\"positive\"] in checked:\n        raise ValueError(f\"{label}: a negative equals the positive\")\n    item[\"negatives\"] = checked\n    if \"intent\" in record:\n        item[\"intent\"] = str(record[\"intent\"])\n    return item\n\n\ndef validate_dataset(\n    records: Sequence[Mapping[str, Any]], *, min_records: int = MIN_RECORDS, max_records: int = MAX_RECORDS\n) -> dict[str, Any]:\n    \"\"\"Structural validation of a query / positive / negatives dataset; raises before any model import.\"\"\"\n    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, (str, bytes)):\n        raise ValueError(\"records must be a list of {id, query, positive, negatives} mappings\")\n    if not min_records <= len(records) <= max_records:\n        raise ValueError(f\"{len(records)} records; {min_records}..{max_records} are required\")\n    checked = []\n    ids: set[str] = set()\n    queries: set[str] = set()\n    for index, record in enumerate(records):\n        item = _check_record(record, index)\n        if item[\"id\"] in ids:\n            raise ValueError(f\"duplicate id {item['id']!r}\")\n        ids.add(item[\"id\"])\n        queries.add(item[\"query\"].lower())\n        checked.append(item)\n    sizes = [1 + len(r[\"negatives\"]) for r in checked]\n    return {\n        \"records\": checked,\n        \"n_records\": len(checked),\n        \"unique_queries\": len(queries),\n        \"n_documents\": len(documents(checked)),\n        \"candidates\": {\"min\": min(sizes), \"max\": max(sizes), \"mean\": sum(sizes) / len(sizes)},\n        \"query_chars\": {\n            \"min\": min(len(r[\"query\"]) for r in checked),\n            \"max\": max(len(r[\"query\"]) for r in checked),\n        },\n        \"digest\": dataset_digest(checked),\n        \"model_id\": MODEL_ID,\n    }\n\n\ndef documents(records: Sequence[Mapping[str, Any]]) -> list[str]:\n    \"\"\"The sorted unique documents (positives and negatives) of a dataset.\"\"\"\n    positives = {str(r[\"positive\"]).strip() for r in records}\n    negatives = {str(n).strip() for r in records for n in r[\"negatives\"]}\n    return sorted(positives | negatives)\n\n\ndef dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:\n    payload = [[r[\"id\"], r[\"query\"], r[\"positive\"], list(r[\"negatives\"])] for r in records]\n    return _sha256_bytes(json.dumps(payload, ensure_ascii=False, separators=(\",\", \":\")).encode(\"utf-8\"))\n\n\ndef check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:\n    \"\"\"Assert no lower-cased query appears in two splits (leakage check).\"\"\"\n    seen: dict[str, str] = {}\n    for name, records in splits.items():\n        for record in records:\n            key = str(record[\"query\"]).lower()\n            if key in seen and seen[key] != name:\n                raise ValueError(f\"query {record['query'][:60]!r} appears in both {seen[key]} and {name}\")\n            seen[key] = name\n    return {name: len(records) for name, records in splits.items()}\n\n\ndef split_dataset(\n    records: Sequence[Mapping[str, Any]],\n    *,\n    val_fraction: float = 0.15,\n    test_fraction: float = 0.2,\n    seed: int = 0,\n) -> dict[str, list[dict[str, Any]]]:\n    \"\"\"Seeded shuffle of a BYOD dataset into train/validation/test after de-duplicating queries.\"\"\"\n    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):\n        raise ValueError(\"fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1\")\n    checked = validate_dataset(records)[\"records\"]\n    seen: set[str] = set()\n    unique = []\n    for record in checked:\n        key = record[\"query\"].lower()\n        if key not in seen:\n            seen.add(key)\n            unique.append(record)\n    random.Random(seed).shuffle(unique)\n    n_test = max(1, round(len(unique) * test_fraction))\n    n_val = round(len(unique) * val_fraction)\n    splits = {\n        \"test\": unique[:n_test],\n        \"validation\": unique[n_test : n_test + n_val],\n        \"train\": unique[n_test + n_val :],\n    }\n    if len(splits[\"train\"]) < MIN_RECORDS:\n        raise ValueError(\n            f\"split leaves {len(splits['train'])} training records; at least {MIN_RECORDS} are required\"\n        )\n    return splits\n\n\ndef load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:\n    \"\"\"Read `{id, query, positive, negatives}` records from CSV (columns id, query, positive, negatives — the\n    negatives separated by ` | `), a JSON array or JSONL (negatives as a list).\"\"\"\n    file_path = Path(path)\n    if not file_path.is_file():\n        raise FileNotFoundError(f\"dataset not found: {file_path}\")\n    suffix = file_path.suffix.lower()\n    text = file_path.read_text(encoding=\"utf-8\")\n    if suffix == \".csv\":\n        rows = list(csv.DictReader(io.StringIO(text)))\n        missing = {\"id\", \"query\", \"positive\", \"negatives\"} - set(rows[0].keys() if rows else set())\n        if missing:\n            raise ValueError(f\"CSV is missing columns {sorted(missing)}\")\n        return [\n            {\n                \"id\": r[\"id\"],\n                \"query\": r[\"query\"],\n                \"positive\": r[\"positive\"],\n                \"negatives\": [\n                    n.strip() for n in r[\"negatives\"].split(NEGATIVE_SEPARATOR.strip()) if n.strip()\n                ],\n            }\n            for r in rows\n        ]\n    if suffix == \".jsonl\":\n        return [json.loads(line) for line in text.splitlines() if line.strip()]\n    if suffix == \".json\":\n        data = json.loads(text)\n        if not isinstance(data, list):\n            raise ValueError(\"JSON dataset must be an array of records\")\n        return data\n    raise ValueError(\"BYOD datasets must be .csv, .json or .jsonl\")\n\n\ndef write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:\n    out = Path(path)\n    out.parent.mkdir(parents=True, exist_ok=True)\n    with open(out, \"w\", encoding=\"utf-8\", newline=\"\") as handle:\n        writer = csv.DictWriter(handle, fieldnames=[\"id\", \"query\", \"positive\", \"negatives\"])\n        writer.writeheader()\n        for record in records:\n            writer.writerow(\n                {\n                    \"id\": record[\"id\"],\n                    \"query\": record[\"query\"],\n                    \"positive\": record[\"positive\"],\n                    \"negatives\": NEGATIVE_SEPARATOR.join(record[\"negatives\"]),\n                }\n            )\n    return out\n"}
for relative,source in EMBEDDED_SOURCES.items():
    path=EMBEDDED_ROOT/relative
    path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(source,encoding="utf-8")

EMBED_WEIGHTS=RUNTIME_ROOT/"weights"/"qwen3-embedding-0.6b"
RERANK_WEIGHTS=RUNTIME_ROOT/"weights"/"qwen3-reranker-0.6b"
EMBED_WEIGHTS.mkdir(parents=True,exist_ok=True)
RERANK_WEIGHTS.mkdir(parents=True,exist_ok=True)
(EMBED_WEIGHTS/"dimer-base-manifest.json").write_text("{\n  \"format\": \"dimer_hf_snapshot\",\n  \"formatVersion\": 1,\n  \"modelKey\": \"qwen3-embedding-0.6b\",\n  \"modelId\": \"Qwen/Qwen3-Embedding-0.6B\",\n  \"revision\": \"97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3\",\n  \"files\": [\n    {\n      \"path\": \"1_Pooling/config.json\",\n      \"bytes\": 313,\n      \"sha256\": \"37bf193fa101f19101bfad9c31d3eb0f786e247b7b1e5cb7f007d730eed1ddbd\"\n    },\n    {\n      \"path\": \"README.md\",\n      \"bytes\": 17237,\n      \"sha256\": \"c34d9b7e5a267ad3fdd13227a253686bc90844ff4744a2a6a86c7c905e3d06f3\"\n    },\n    {\n      \"path\": \"config.json\",\n      \"bytes\": 727,\n      \"sha256\": \"b5bf1f51fc45be473a54718cef92448d90a1be001bf9b9a44b8c7f10a19feaa9\"\n    },\n    {\n      \"path\": \"config_sentence_transformers.json\",\n      \"bytes\": 215,\n      \"sha256\": \"10667c72ddb772627bf1780cb7f86af8e2ae0032b8c243c731172064105c6961\"\n    },\n    {\n      \"path\": \"generation_config.json\",\n      \"bytes\": 117,\n      \"sha256\": \"28396d421a2108acce96383f6a7de78008f7f1b17f807958f3c14c51dbfb65fb\"\n    },\n    {\n      \"path\": \"merges.txt\",\n      \"bytes\": 1671853,\n      \"sha256\": \"8831e4f1a044471340f7c0a83d7bd71306a5b867e95fd870f74d0c5308a904d5\"\n    },\n    {\n      \"path\": \"model.safetensors\",\n      \"bytes\": 1191586416,\n      \"sha256\": \"0437e45c94563b09e13cb7a64478fc406947a93cb34a7e05870fc8dcd48e23fd\"\n    },\n    {\n      \"path\": \"modules.json\",\n      \"bytes\": 349,\n      \"sha256\": \"84e40c8e006c9b1d6c122e02cba9b02458120b5fb0c87b746c41e0207cf642cf\"\n    },\n    {\n      \"path\": \"tokenizer.json\",\n      \"bytes\": 11423705,\n      \"sha256\": \"def76fb086971c7867b829c23a26261e38d9d74e02139253b38aeb9df8b4b50a\"\n    },\n    {\n      \"path\": \"tokenizer_config.json\",\n      \"bytes\": 9706,\n      \"sha256\": \"253153d0738ceb4c668d2eff957714dd2bea0b56de772a9fdccd96cbf517e6a0\"\n    },\n    {\n      \"path\": \"vocab.json\",\n      \"bytes\": 2776833,\n      \"sha256\": \"ca10d7e9fb3ed18575dd1e277a2579c16d108e32f27439684afa0e10b1440910\"\n    }\n  ],\n  \"totalBytes\": 1207487471\n}",encoding="utf-8")
(RERANK_WEIGHTS/"dimer-base-manifest.json").write_text("{\n  \"format\": \"dimer_hf_snapshot\",\n  \"formatVersion\": 1,\n  \"modelKey\": \"qwen3-reranker-0.6b\",\n  \"modelId\": \"Qwen/Qwen3-Reranker-0.6B\",\n  \"revision\": \"e61197ed45024b0ed8a2d74b80b4d909f1255473\",\n  \"files\": [\n    {\n      \"path\": \"1_LogitScore/config.json\",\n      \"bytes\": 57,\n      \"sha256\": \"73e3156450564d8a98b7e47bcf5aace0f29600828b51937da545571e84db3ff3\"\n    },\n    {\n      \"path\": \"README.md\",\n      \"bytes\": 14742,\n      \"sha256\": \"5bba8c734f6dd3ae48317b4139317e45a7fce48fc55e15670b23a0dd15492ab6\"\n    },\n    {\n      \"path\": \"chat_template.jinja\",\n      \"bytes\": 741,\n      \"sha256\": \"6f682162495ec5b39fd9005c01b6aa2a74669379fe967039f1e2cbbe8752369d\"\n    },\n    {\n      \"path\": \"config.json\",\n      \"bytes\": 727,\n      \"sha256\": \"d479c427a9ca5295218063d4f9aca4f297ab4ac27487cca7af42c84643d51ef0\"\n    },\n    {\n      \"path\": \"config_sentence_transformers.json\",\n      \"bytes\": 325,\n      \"sha256\": \"6a153d6696f78fd588c1c728967f0b773ea869d3c6028f151ce71ebe49140762\"\n    },\n    {\n      \"path\": \"generation_config.json\",\n      \"bytes\": 214,\n      \"sha256\": \"81051cd3f6e77013827148d0b8a6ead93f8ac390d5ab805f849199f0af6a08db\"\n    },\n    {\n      \"path\": \"merges.txt\",\n      \"bytes\": 1671853,\n      \"sha256\": \"8831e4f1a044471340f7c0a83d7bd71306a5b867e95fd870f74d0c5308a904d5\"\n    },\n    {\n      \"path\": \"model.safetensors\",\n      \"bytes\": 1191588280,\n      \"sha256\": \"27cd75a405b9c1b46b59abfd88aaa209e6fed2a1972cde9b70e7659537c5e65b\"\n    },\n    {\n      \"path\": \"modules.json\",\n      \"bytes\": 280,\n      \"sha256\": \"6f13b6b4a89e577b591b2077bca40c67c26541a6740a8809267cb474f90806a9\"\n    },\n    {\n      \"path\": \"sentence_bert_config.json\",\n      \"bytes\": 362,\n      \"sha256\": \"3234ebd224d492cbe8d55d5ec80a3f408451c4db3005bafb64fe1c51c763e01e\"\n    },\n    {\n      \"path\": \"tokenizer.json\",\n      \"bytes\": 11422654,\n      \"sha256\": \"aeb13307a71acd8fe81861d94ad54ab689df773318809eed3cbe794b4492dae4\"\n    },\n    {\n      \"path\": \"tokenizer_config.json\",\n      \"bytes\": 9706,\n      \"sha256\": \"253153d0738ceb4c668d2eff957714dd2bea0b56de772a9fdccd96cbf517e6a0\"\n    },\n    {\n      \"path\": \"vocab.json\",\n      \"bytes\": 2776833,\n      \"sha256\": \"ca10d7e9fb3ed18575dd1e277a2579c16d108e32f27439684afa0e10b1440910\"\n    }\n  ],\n  \"totalBytes\": 1207486774\n}",encoding="utf-8")

SOURCE_PROVENANCE={
    "embedding":{"repository":"kurtvalcorza/qwen3-embedding-pipeline","commit":"115cf17fb35048dcadc187ac7dd36b28d99f9aab"},
    "reranker":{"repository":"kurtvalcorza/qwen3-reranker-pipeline","commit":"f13a58e65a7ee54343e8fa262c166308699c4f11"},
}
print({"embedded_files":len(EMBEDDED_SOURCES),"source_provenance":SOURCE_PROVENANCE})


## 2. Dataset and BYOD contract

### Default dataset

The default path uses the pinned **Banking77** corpus already carried by both live DIMER profiles.

- 77 fine-grained banking intents
- each intent phrase is one candidate document, e.g. `card_arrival` → `card arrival`
- each customer-support message is a query
- the correct intent phrase is the single relevant document
- the workshop evaluates the existing **385-query balanced test sample** (five queries per intent)
- corpus licence: **CC BY 4.0**

The upstream CSVs are pinned by byte count and SHA-256 by the embedded sample module before parsing.

### BYOD

Set `USE_BYOD=True` and point `BYOD_ZIP_PATH` to a local ZIP containing exactly:

**documents.csv**

`document_id,text`

**queries.csv**

`query_id,query,relevant_document_id`

Every query must name exactly one document in `documents.csv`. IDs and document texts must be unique. The workshop currently supports **2–1000 documents** and **10–2000 queries**.

This schema represents a single-relevant-document retrieval task. If your domain has multiple or graded relevant documents, extend the judgement schema and use appropriate multi-relevance nDCG/recall calculations rather than forcing it into this exercise.


In [ ]:
# @title 2.1 Validate and extract BYOD before model work
MAX_ARCHIVE_EXPANDED_BYTES=512*1024**2
BYOD_ROOT=None

def safe_extract_zip(path,destination):
    destination.mkdir(parents=True,exist_ok=True)
    seen=set()
    expanded=0
    with zipfile.ZipFile(path) as archive:
        for info in archive.infolist():
            member=PurePosixPath(info.filename)
            if member.is_absolute() or ".." in member.parts:
                raise ValueError(f"Unsafe archive path: {info.filename!r}")
            if info.filename in seen:
                raise ValueError(f"Duplicate archive member: {info.filename!r}")
            seen.add(info.filename)
            mode=(info.external_attr>>16)&0o170000
            if mode==stat.S_IFLNK:
                raise ValueError(f"Symlinks are not allowed: {info.filename!r}")
            expanded+=info.file_size
            if expanded>MAX_ARCHIVE_EXPANDED_BYTES:
                raise ValueError("Archive expands beyond the 512 MiB workshop ceiling.")
        archive.extractall(destination)
    return destination

def validate_byod_root(root):
    docs=list(root.rglob("documents.csv"))
    queries=list(root.rglob("queries.csv"))
    if len(docs)!=1 or len(queries)!=1 or docs[0].parent!=queries[0].parent:
        raise ValueError("BYOD ZIP must contain one documents.csv and one queries.csv in the same directory.")
    root=docs[0].parent
    d=pd.read_csv(docs[0])
    q=pd.read_csv(queries[0])
    if list(d.columns)!=["document_id","text"]:
        raise ValueError("documents.csv columns must be exactly document_id,text.")
    if list(q.columns)!=["query_id","query","relevant_document_id"]:
        raise ValueError("queries.csv columns must be exactly query_id,query,relevant_document_id.")
    if not 2<=len(d)<=1000:
        raise ValueError(f"documents.csv must contain 2..1000 rows; got {len(d)}.")
    if not 10<=len(q)<=2000:
        raise ValueError(f"queries.csv must contain 10..2000 rows; got {len(q)}.")
    if d["document_id"].astype(str).duplicated().any() or d["text"].astype(str).duplicated().any():
        raise ValueError("document IDs and document texts must each be unique.")
    if q["query_id"].astype(str).duplicated().any():
        raise ValueError("query IDs must be unique.")
    if d[["document_id","text"]].isna().any().any() or q.isna().any().any():
        raise ValueError("BYOD cells may not be empty.")
    valid=set(d["document_id"].astype(str))
    missing=sorted(set(q["relevant_document_id"].astype(str))-valid)
    if missing:
        raise ValueError(f"queries reference unknown document IDs: {missing[:5]}")
    if CANDIDATE_K>=len(d):
        raise ValueError(f"CANDIDATE_K={CANDIDATE_K} must be smaller than the {len(d)}-document corpus.")
    return root,{"documents":len(d),"queries":len(q)}

if USE_BYOD:
    if not BYOD_ZIP_PATH:
        raise ValueError("USE_BYOD=True requires BYOD_ZIP_PATH.")
    archive=Path(BYOD_ZIP_PATH)
    if not archive.is_file():
        raise FileNotFoundError(archive)
    extracted=safe_extract_zip(archive,RUNTIME_ROOT/"byod")
    BYOD_ROOT,byod_summary=validate_byod_root(extracted)
    print({"byod_root":str(BYOD_ROOT),**byod_summary})
else:
    print("Using the pinned Banking77 test sample; no upload dialog is opened.")


## 3. Retrieval and reranking metrics

For each query we produce three full-corpus rankings:

1. **Lexical floor** — Jaccard overlap between query tokens and document tokens.
2. **Embedding-only** — cosine similarity between Qwen3 query/document embeddings.
3. **Two-stage** — take the embedding model's top `CANDIDATE_K`, rerank only those with Qwen3-Reranker, then append the untouched embedding tail.

The third construction matters. It lets us compare end-to-end rankings without pretending the reranker evaluated all 77 documents.

Metrics:

- **Recall@1/5/10** — whether the relevant document appears in the first k positions.
- **MRR** — mean reciprocal rank of the relevant document.
- **nDCG@10** — for this one-relevant-document task, discounted gain of the relevant document if it appears in the top 10.
- **Median rank** — a robust view of ranking position.
- **Candidate recall@K** — fraction of queries whose relevant document was present in the reranker's shortlist. This is the reranker's hard ceiling for recovering retrieval misses.
- **Conditional reranker top-1** — among queries whose relevant document *was* in the shortlist, how often reranking placed it first.

The reranker's numeric relevance score is used only for ordering within one query's candidates; it is not treated as a calibrated probability or compared across unrelated queries.


In [ ]:
# @title 3.1 Install the shared pinned runtime
RUNTIME_PINS=[
    "torch==2.14.0",
    "torchvision==0.29.0",
    "torchaudio==2.11.0",
    "transformers==4.57.6",
    "huggingface-hub==0.36.2",
    "safetensors==0.8.0",
    "numpy==2.5.3",
]
ENV_DIR=RUNTIME_ROOT/"retrieval_env"
PYTHON=ENV_DIR/("Scripts/python.exe" if os.name=="nt" else "bin/python")
marker=ENV_DIR/"dimer_pins.json"
pins_text=json.dumps(RUNTIME_PINS,sort_keys=True)
if not PYTHON.exists():
    venv.EnvBuilder(with_pip=True).create(ENV_DIR)
if not marker.exists() or marker.read_text()!=pins_text:
    subprocess.run([str(PYTHON),"-m","pip","install","--disable-pip-version-check","-q",*RUNTIME_PINS],check=True)
    marker.write_text(pins_text)
print({"runtime_python":str(PYTHON),"pins":RUNTIME_PINS})


## 4. Execute the two-stage pipeline

The runner validates the query/document task **before loading either 0.6B model**.

It then:

1. loads the embedding model from its manifest-verified snapshot;
2. embeds the document corpus once;
3. embeds all queries and obtains the full cosine ranking;
4. records the top-K candidate set;
5. frees the embedding model and GPU cache;
6. loads the reranker from its manifest-verified snapshot;
7. scores only the retrieved candidate pairs; and
8. constructs the final two-stage ranking.

This sequential loading keeps peak GPU memory lower than keeping both 0.6B models resident.


In [ ]:
# @title 4.1 Materialize and execute the standalone retrieval runner
RUNNER_SOURCE=r"""
import csv
import gc
import json
import math
import platform
import re
import sys
import time
from pathlib import Path

import numpy as np
import torch

cfg=json.loads(Path(sys.argv[1]).read_text())
out_path=Path(sys.argv[2])
sys.path.insert(0,cfg["embedded_root"])

from qwen3_embedding_pipeline import Qwen3EmbeddingPipeline
from qwen3_embedding_pipeline import metrics as emb_metrics
from qwen3_embedding_pipeline import samples as emb_samples
from qwen3_reranker_pipeline import Qwen3RerankerPipeline
from qwen3_reranker_pipeline import samples as rer_samples

device="cuda:0" if torch.cuda.is_available() else "cpu"
if not device.startswith("cuda"):
    raise RuntimeError("CUDA is required by the reference two-stage workshop path.")

def validate_text(value,label,ceiling=100000):
    value=str(value).strip()
    if not value:
        raise ValueError(f"{label} is empty")
    if len(value)>ceiling:
        raise ValueError(f"{label} exceeds {ceiling} characters")
    return value

def load_task():
    if cfg["use_byod"]:
        root=Path(cfg["byod_root"])
        docs_rows=list(csv.DictReader((root/"documents.csv").open(encoding="utf-8")))
        query_rows=list(csv.DictReader((root/"queries.csv").open(encoding="utf-8")))
        docs=[]
        doc_ids=[]
        id_to_text={}
        for i,row in enumerate(docs_rows):
            did=validate_text(row["document_id"],f"documents[{i}].document_id",1000)
            text=validate_text(row["text"],f"documents[{i}].text")
            if did in id_to_text:
                raise ValueError(f"duplicate document_id {did!r}")
            if text in docs:
                raise ValueError("document texts must be unique")
            id_to_text[did]=text
            doc_ids.append(did)
            docs.append(text)
        queries=[]
        seen=set()
        for i,row in enumerate(query_rows):
            qid=validate_text(row["query_id"],f"queries[{i}].query_id",1000)
            query=validate_text(row["query"],f"queries[{i}].query")
            relevant=validate_text(row["relevant_document_id"],f"queries[{i}].relevant_document_id",1000)
            if qid in seen:
                raise ValueError(f"duplicate query_id {qid!r}")
            if relevant not in id_to_text:
                raise ValueError(f"unknown relevant_document_id {relevant!r}")
            seen.add(qid)
            queries.append({"id":qid,"query":query,"positive":id_to_text[relevant],"relevant_document_id":relevant})
        instruction=cfg["byod_instruction"]
        sample_kind="BYOD"
    else:
        splits=emb_samples.fetch_sample_dataset(cache_dir=Path(cfg["runtime_root"])/"banking77")
        records=emb_samples.validate_dataset(splits["test"],min_records=50,max_records=2000)["records"]
        docs=emb_samples.documents(records)
        doc_ids=[f"intent-{i:03d}" for i in range(len(docs))]
        text_to_id=dict(zip(docs,doc_ids,strict=True))
        queries=[
            {"id":r["id"],"query":r["query"],"positive":r["positive"],"relevant_document_id":text_to_id[r["positive"]]}
            for r in records[:int(cfg["max_queries"])]
        ]
        instruction=emb_samples.DEFAULT_INSTRUCTION
        sample_kind="Banking77 pinned test sample"
    if not 2<=len(docs)<=1000:
        raise ValueError(f"document corpus must hold 2..1000 items; got {len(docs)}")
    if not 10<=len(queries)<=2000:
        raise ValueError(f"query set must hold 10..2000 items; got {len(queries)}")
    if int(cfg["candidate_k"])>=len(docs):
        raise ValueError("candidate_k must be smaller than the document corpus")
    index={text:i for i,text in enumerate(docs)}
    if len(index)!=len(docs):
        raise ValueError("document texts must be unique")
    missing=[q["positive"] for q in queries if q["positive"] not in index]
    if missing:
        raise ValueError(f"positive document absent from corpus: {missing[0]!r}")
    return docs,doc_ids,queries,instruction,sample_kind,index

def embed_all(pipe,texts,kind,instruction):
    rows=[]
    for start in range(0,len(texts),64):
        rows.extend(pipe.embed(texts[start:start+64],kind=kind,instruction=instruction)["embeddings"])
    return np.asarray(rows,dtype=np.float32)

def aggregate(ranks,n_documents):
    ranks=[int(r) for r in ranks]
    if any(r<1 or r>n_documents for r in ranks):
        raise ValueError("rank outside corpus")
    out={"n_queries":len(ranks),"n_documents":n_documents}
    for k in (1,5,10):
        out[f"recall@{k}"]=float(np.mean([r<=min(k,n_documents) for r in ranks]))
    out["mrr"]=float(np.mean([1.0/r for r in ranks]))
    out["median_rank"]=float(np.median(ranks))
    out["ndcg@10"]=float(np.mean([1.0/math.log2(r+1) if r<=10 else 0.0 for r in ranks]))
    return out

docs,doc_ids,queries,embed_instruction,sample_kind,index=load_task()
rerank_instruction=(
    "Given a customer support message, judge whether the document names the banking intent it expresses"
    if not cfg["use_byod"] else cfg["byod_rerank_instruction"]
)

# Lexical floor.
lexical_ranks=[]
lexical_orders=[]
for q in queries:
    scores=[emb_metrics.jaccard(q["query"],doc) for doc in docs]
    positive_index=index[q["positive"]]
    lexical_ranks.append(emb_metrics.rank_of_positive(scores,positive_index))
    lexical_orders.append(np.argsort(-np.asarray(scores),kind="stable").tolist())
lexical_metrics=aggregate(lexical_ranks,len(docs))

# Stage 1: embeddings.
embed_started=time.perf_counter()
embedder=Qwen3EmbeddingPipeline.from_pretrained(
    device=device,weights_dir=cfg["embedding_weights"],allow_download=True
)
doc_vectors=embed_all(embedder,docs,"document",embed_instruction)
query_vectors=embed_all(embedder,[q["query"] for q in queries],"query",embed_instruction)
scores=query_vectors@doc_vectors.T
embed_orders=np.argsort(-scores,axis=1,kind="stable")
embed_ranks=[]
for row,q in zip(scores,queries,strict=True):
    embed_ranks.append(emb_metrics.rank_of_positive(row.tolist(),index[q["positive"]]))
embed_seconds=time.perf_counter()-embed_started
embedding_metrics=aggregate(embed_ranks,len(docs))

candidate_k=int(cfg["candidate_k"])
candidate_orders=[row[:candidate_k].tolist() for row in embed_orders]
candidate_contains=[index[q["positive"]] in candidates for q,candidates in zip(queries,candidate_orders,strict=True)]
candidate_recall=float(np.mean(candidate_contains))

# Free the first 0.6B model before loading the second.
del embedder
gc.collect()
torch.cuda.empty_cache()

# Stage 2: rerank only candidate pairs.
rerank_started=time.perf_counter()
reranker=Qwen3RerankerPipeline.from_pretrained(
    device=device,weights_dir=cfg["reranker_weights"],allow_download=True
)
flat_pairs=[]
for q,candidates in zip(queries,candidate_orders,strict=True):
    flat_pairs.extend((q["query"],docs[i]) for i in candidates)
flat_scores=[]
for start in range(0,len(flat_pairs),32):
    flat_scores.extend(reranker.rerank(flat_pairs[start:start+32],instruction=rerank_instruction)["scores"])

final_orders=[]
rerank_ranks=[]
conditional_top1=[]
cursor=0
for q,embedding_order,candidates,contains in zip(queries,embed_orders,candidate_orders,candidate_contains,strict=True):
    local=np.asarray(flat_scores[cursor:cursor+len(candidates)],dtype=float)
    cursor+=len(candidates)
    candidate_rank=np.argsort(-local,kind="stable")
    reranked=[candidates[int(i)] for i in candidate_rank]
    candidate_set=set(candidates)
    tail=[int(i) for i in embedding_order.tolist() if int(i) not in candidate_set]
    final=reranked+tail
    final_orders.append(final)
    positive=index[q["positive"]]
    rerank_ranks.append(final.index(positive)+1)
    if contains:
        conditional_top1.append(reranked[0]==positive)
rerank_seconds=time.perf_counter()-rerank_started
two_stage_metrics=aggregate(rerank_ranks,len(docs))
two_stage_metrics["candidate_recall@k"]=candidate_recall
two_stage_metrics["candidate_k"]=candidate_k
two_stage_metrics["conditional_reranker_top1"]=float(np.mean(conditional_top1)) if conditional_top1 else None
two_stage_metrics["queries_with_positive_in_candidates"]=int(sum(candidate_contains))

random_metrics={
    "recall@1":1/len(docs),
    "recall@5":min(5,len(docs))/len(docs),
    "recall@10":min(10,len(docs))/len(docs),
    "mrr":sum(1/r for r in range(1,len(docs)+1))/len(docs),
    "ndcg@10":sum(1/math.log2(r+1) for r in range(1,min(10,len(docs))+1))/len(docs),
}

per_query=[]
for i,q in enumerate(queries):
    relevant=index[q["positive"]]
    per_query.append({
        "query_id":q["id"],
        "query":q["query"],
        "relevant_document_id":q["relevant_document_id"],
        "relevant_document":q["positive"],
        "candidate_contains_positive":bool(candidate_contains[i]),
        "lexical_rank":int(lexical_ranks[i]),
        "embedding_rank":int(embed_ranks[i]),
        "two_stage_rank":int(rerank_ranks[i]),
        "embedding_top1":docs[int(embed_orders[i][0])],
        "two_stage_top1":docs[int(final_orders[i][0])],
        "candidate_documents":[docs[j] for j in candidate_orders[i]],
    })

out={
    "sample_kind":sample_kind,
    "instructions":{"embedding":embed_instruction,"reranker":rerank_instruction},
    "runtime":{"python":platform.python_version(),"torch":torch.__version__,"numpy":np.__version__,"device":device},
    "models":{
        "embedding":{"id":"Qwen/Qwen3-Embedding-0.6B","revision":"97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3"},
        "reranker":{"id":"Qwen/Qwen3-Reranker-0.6B","revision":"e61197ed45024b0ed8a2d74b80b4d909f1255473"},
    },
    "n_documents":len(docs),
    "n_queries":len(queries),
    "candidate_k":candidate_k,
    "metrics":{
        "random_expected":random_metrics,
        "lexical":lexical_metrics,
        "embedding":embedding_metrics,
        "two_stage":two_stage_metrics,
    },
    "timing_seconds":{"embedding_stage":embed_seconds,"reranking_stage":rerank_seconds},
    "documents":[{"document_id":doc_ids[i],"text":doc} for i,doc in enumerate(docs)],
    "per_query":per_query,
}
out_path.write_text(json.dumps(out,indent=2,ensure_ascii=False))
"""
runner_path=RUNTIME_ROOT/"semantic_search_runner.py"
runner_path.write_text(RUNNER_SOURCE,encoding="utf-8")
runner_cfg={
    "embedded_root":str(EMBEDDED_ROOT),
    "runtime_root":str(RUNTIME_ROOT),
    "embedding_weights":str(EMBED_WEIGHTS),
    "reranker_weights":str(RERANK_WEIGHTS),
    "use_byod":USE_BYOD,
    "byod_root":str(BYOD_ROOT) if BYOD_ROOT else None,
    "candidate_k":CANDIDATE_K,
    "max_queries":MAX_QUERIES,
    "byod_instruction":"Given a search query, retrieve the document that best satisfies it",
    "byod_rerank_instruction":"Given a search query, judge whether the document satisfies the query",
}
config_path=RUNTIME_ROOT/"semantic_search_runner_config.json"
config_path.write_text(json.dumps(runner_cfg,indent=2))
result_path=RUNTIME_ROOT/"semantic_search_results.json"

started=time.perf_counter()
run=subprocess.run([str(PYTHON),str(runner_path),str(config_path),str(result_path)],text=True,capture_output=True)
WALL_SECONDS=time.perf_counter()-started
if run.returncode!=0:
    print(run.stdout)
    print(run.stderr)
    raise RuntimeError(f"Semantic-search runner failed with exit code {run.returncode}")
RUN_RESULTS=json.loads(result_path.read_text())
print({
    "sample_kind":RUN_RESULTS["sample_kind"],
    "documents":RUN_RESULTS["n_documents"],
    "queries":RUN_RESULTS["n_queries"],
    "candidate_k":RUN_RESULTS["candidate_k"],
    "wall_seconds":round(WALL_SECONDS,2),
})


## 5. Compare the three ranking stages

The table below keeps the roles separate:

- **random expected** is a mathematical floor, not a run;
- **lexical** uses no neural model;
- **embedding** ranks all documents by dense cosine similarity;
- **two-stage** reranks only the embedding top-K.

A useful two-stage system can improve top-rank quality while preserving the retriever's candidate-recall ceiling. If candidate recall@K is low, the first stage—not the reranker—is the bottleneck.


In [ ]:
# @title 5.1 Ranking metrics
metric_rows=[]
for stage,values in RUN_RESULTS["metrics"].items():
    metric_rows.append({
        "stage":stage,
        "recall@1":values.get("recall@1"),
        "recall@5":values.get("recall@5"),
        "recall@10":values.get("recall@10"),
        "mrr":values.get("mrr"),
        "ndcg@10":values.get("ndcg@10"),
        "median_rank":values.get("median_rank"),
        "candidate_recall@k":values.get("candidate_recall@k"),
        "conditional_reranker_top1":values.get("conditional_reranker_top1"),
    })
metrics=pd.DataFrame(metric_rows)
display(metrics)

print({
    "candidate_k":RUN_RESULTS["candidate_k"],
    "candidate_recall":RUN_RESULTS["metrics"]["two_stage"]["candidate_recall@k"],
    "queries_positive_reached_reranker":RUN_RESULTS["metrics"]["two_stage"]["queries_with_positive_in_candidates"],
    "conditional_reranker_top1":RUN_RESULTS["metrics"]["two_stage"]["conditional_reranker_top1"],
})


## 6. Analyze when reranking helps—and when it cannot

The per-query table records three ranks. Two derived groups are especially useful:

- **rescued:** embedding ranked the relevant document below position 1, but the two-stage system moved it to rank 1;
- **retrieval miss:** the relevant document was not in the top-K candidate set, so reranking never saw it.

A reranker should not be blamed for a retrieval miss, and a retriever should not receive credit for ordering improvements produced by the second stage.


In [ ]:
# @title 6.1 Per-query failure decomposition
per_query=pd.DataFrame(RUN_RESULTS["per_query"])
per_query["rescued_to_top1"]=(per_query["embedding_rank"]>1)&(per_query["two_stage_rank"]==1)
per_query["hurt_from_top1"]=(per_query["embedding_rank"]==1)&(per_query["two_stage_rank"]>1)
per_query["rank_delta"]=per_query["embedding_rank"]-per_query["two_stage_rank"]

summary={
    "rescued_to_top1":int(per_query["rescued_to_top1"].sum()),
    "hurt_from_top1":int(per_query["hurt_from_top1"].sum()),
    "retrieval_misses_outside_candidate_k":int((~per_query["candidate_contains_positive"]).sum()),
    "mean_rank_improvement_positive_is_better":float(per_query["rank_delta"].mean()),
}
print(summary)
display(
    per_query.sort_values(["candidate_contains_positive","rank_delta"],ascending=[True,False])
    .head(SHOW_EXAMPLES)[
        ["query_id","query","relevant_document","candidate_contains_positive","embedding_rank","two_stage_rank","embedding_top1","two_stage_top1","rank_delta"]
    ]
)


## 7. Quality-versus-cost view

Dense retrieval is efficient because the document corpus is embedded once and query-document similarity is a matrix operation. Cross-encoder reranking is more expensive because the model jointly reads each query-candidate pair.

The timing bars below are hardware- and cache-specific; they are useful for understanding the architecture of the workflow, not for publishing portable latency claims.


In [ ]:
# @title 7.1 Stage timing and metric change
timing=pd.DataFrame([
    {"stage":"embedding retrieval","seconds":RUN_RESULTS["timing_seconds"]["embedding_stage"]},
    {"stage":f"rerank top-{CANDIDATE_K}","seconds":RUN_RESULTS["timing_seconds"]["reranking_stage"]},
])
display(timing)

fig=plt.figure(figsize=(7,4))
plt.bar(timing["stage"],timing["seconds"])
plt.ylabel("seconds in this run")
plt.title("Measured stage wall time")
plt.xticks(rotation=15)
plt.tight_layout()
timing_plot=OUTPUT_ROOT/"semantic_search_stage_runtime.png"
fig.savefig(timing_plot,dpi=140)
plt.show()


## 8. Export rankings and provenance

The notebook exports:

- `semantic_search_metrics.csv`
- `semantic_search_per_query.csv`
- `semantic_search_documents.csv`
- `semantic_search_provenance.json`
- the runtime chart

The per-query CSV is the most useful audit surface because it shows whether each relevant document reached the reranker and how its rank changed.


In [ ]:
# @title 8.1 Machine-readable exports
metrics.to_csv(OUTPUT_ROOT/"semantic_search_metrics.csv",index=False)
per_query.drop(columns=["candidate_documents"]).to_csv(OUTPUT_ROOT/"semantic_search_per_query.csv",index=False)
pd.DataFrame(RUN_RESULTS["documents"]).to_csv(OUTPUT_ROOT/"semantic_search_documents.csv",index=False)

provenance={
    "notebook":{
        "specification":"2.1",
        "profile":"TASK-INFERENCE",
        "mode":"WORKSHOP",
        "standalone":True,
        "host_repository":"kurtvalcorza/qwen3-embedding-pipeline",
        "generator":"tools/build_semantic_search_reranking_workshop.py",
        "clean_runtime_evidence":"pending",
    },
    "source_provenance":SOURCE_PROVENANCE,
    "models":RUN_RESULTS["models"],
    "instructions":RUN_RESULTS["instructions"],
    "dataset":{
        "sample_kind":RUN_RESULTS["sample_kind"],
        "documents":RUN_RESULTS["n_documents"],
        "queries":RUN_RESULTS["n_queries"],
        "candidate_k":RUN_RESULTS["candidate_k"],
        "default_corpus_license":"CC BY 4.0" if not USE_BYOD else None,
    },
    "runtime":{**RUN_RESULTS["runtime"],"gpu":GPU_INFO,"wall_seconds":WALL_SECONDS,**RUN_RESULTS["timing_seconds"]},
}
(OUTPUT_ROOT/"semantic_search_provenance.json").write_text(json.dumps(provenance,indent=2),encoding="utf-8")
print(sorted(path.name for path in OUTPUT_ROOT.iterdir()))


## 9. Interpretation and limits

This workshop demonstrates a standard two-stage retrieval pattern with two pinned live DIMER models. It does **not** establish a general production search benchmark.

Limitations:

- Banking77 reduces each intent to a short phrase and gives every query exactly one relevant document;
- the candidate corpus contains only 77 documents;
- real search systems often have multiple or graded relevant documents;
- the embedding model and reranker were not adapted in this workshop;
- one test sample gives no dispersion estimate;
- lexical overlap is a deliberately simple baseline rather than a tuned BM25 implementation;
- the reranker score is used only for within-query ordering and is not a calibrated probability;
- the measured latency is specific to this GPU, software image, cache state, and corpus size; and
- a production system needs larger-corpus indexing, multi-relevance judgements, throughput/concurrency tests, and domain-specific safety/privacy review.

### Suggested exercises

1. Change `CANDIDATE_K` to 5 or 20. How does candidate recall trade off against reranking cost?
2. Inspect retrieval misses. Are the relevant intent phrases semantically ambiguous or simply absent from the embedding top-K?
3. Use BYOD with a larger internal knowledge-base sample and judged queries.
4. Replace the simple lexical floor with BM25 under the same judgements.
5. **Optional third stage:** feed the top retrieved passages to Qwen3-0.6B or SmolLM2-360M-Instruct and evaluate answer grounding separately. Do not infer answer quality from retrieval metrics alone.

### References

- DIMER Notebook Specification 2.1: `ml-worker/integrations/dimer/fleet-specs/NOTEBOOK_SPEC.md`
- DIMER model fleet: `ml-worker/integrations/dimer/fleet-inventory/MODEL_MATRIX.md`
- Banking77: Casanueva et al. (2020), CC BY 4.0
- Qwen3-Embedding-0.6B and Qwen3-Reranker-0.6B model repositories


In [ ]:
# @title Run-all completion summary
summary={
    "notebook_spec":"2.1",
    "profile":"TASK-INFERENCE",
    "mode":"WORKSHOP",
    "pipeline":"Qwen3-Embedding-0.6B retrieval -> Qwen3-Reranker-0.6B reranking",
    "documents":RUN_RESULTS["n_documents"],
    "queries":RUN_RESULTS["n_queries"],
    "candidate_k":RUN_RESULTS["candidate_k"],
    "outputs":sorted(path.name for path in OUTPUT_ROOT.iterdir()),
    "clean_runtime_evidence":"pending until exact hosted-runtime execution is recorded",
}
display(pd.Series(summary,name="value").to_frame())
print("Run-all complete: task validation, full-corpus dense retrieval, candidate reranking, end-to-end ranking evaluation, failure decomposition, and export succeeded.")
